# NB07V3 v1 -- Dietrich Pipeline & Geographic Leakage Analysis (V3 per-point sample unit)

> Copyright (C) 2024-2026 Marco Heinzen - SPDX-License-Identifier: AGPL-3.0-or-later
> Part of the Master Thesis "Building Damage Assessment with Multimodal Satellite Time Series and Machine Learning in the Russia-Ukraine War 2022-2026"
> Code hosted at https://github.com/marcoheinzen/bda
> Parts of this code were written or improved with the assistance of Claude (Anthropic); all other code, and the concept, research, architecture, design, execution, testing and validation throughout, are the author's work.



| Field | Value |
|---|---|
| Notebook | `07V3_BDA_Dietrich_Pipeline_v1.ipynb` |
| Version | `V3 v1` |
| Pipeline position | After NB05bV3 v6, parallel to NB07V2 v6. Consumes V3 (per-point) parquets. |
| Last validated | `2026-04-29` |

## NB07V3 v1 vs NB07V2 v6

Same Dietrich replication pipeline, same RF classifier (n_estimators=200, balanced,
StratifiedKFold + GroupKFold), same diagnostic cells (D1-D11), same OOF prediction
schema. The single change is the sample unit:

- **NB07V2:** one row per (city, building_id), V2 footprint parquets from `dataset/v2/`.
- **NB07V3:** one row per (city, point_id), V3 per-point parquets from `dataset/v3/`.

This is the true Dietrich replication at per-pixel granularity. Dietrich 2025
operates at pixel level (not footprint), so V3's single-pixel sampling more
closely matches the Dietrich methodology than V2's footprint aggregation. The
V2 approach produces Dietrich-equivalent AUCs (0.823 on block_stats) via a
different data path (footprint zonal stats instead of per-pixel sampling); V3
produces the same data path as Dietrich but with UNOSAT labels instead of
Copernicus EMS grading points.

### Changes from NB07V2 v6 (purely surgical)

- Sample-unit column: `'building_id'` -> `'point_id'`.
- Dataset path: `STACK_DIR / 'dataset' / 'v2'` -> `STACK_DIR / 'dataset' / 'v3'`.
- Variable names: `DATASET_ROOT_V2` -> `DATASET_ROOT_V3`, `V2_DIR` -> `V3_DIR`,
  `load_v2_parquet` -> `load_v3_parquet`, `df_buildings` -> `df_points`.
- Sample-unit parquet: `bda_buildings` -> `bda_points`.
- Output dir: `RESULTS_ROOT / 'nb07'` -> `RESULTS_ROOT / 'nb07v3'`.
- OOF schema: `building_id` column -> `point_id` column.
- Centroid references: `centroid_x`/`centroid_y` -> `x_utm`/`y_utm` fallback chain.

In [1]:
# @title CELL 1: NB07V3 v1 CONFIG
TIER_SELECTION = [0,1,2]
CITY_SELECTION = None
TARGET_COL = 'damage_binary'
RANDOM_STATE = 42
FILTER_UNOSAT_ONLY = True
RF_PARAMS = {'n_estimators': 200, 'min_samples_leaf': 3, 'random_state': 42, 'n_jobs': -1, 'class_weight': 'balanced'}
DIETRICH_PAPER = {'auc': 0.813, 'f1': 0.749, 'precision': 0.671, 'recall': 0.846}

# Leakage exclusion patterns (from NB06 v5 REVIEW)
import re
EXCLUDE_PATTERNS = [
    r's1__coh__scenes_observed',
    r's2__obs_count__',
    r's2__qa__cloud_freq__',
    r's2__visibility__cloud__freq',
]

def exclude_leakage(feat_cols):
    return [c for c in feat_cols if not any(re.match(p, c) for p in EXCLUDE_PATTERNS)]


In [2]:
# @title CELL 4: LOAD GLOBAL SETUP
import platform, os, json
if platform.system() == "Windows":
    _setup = r"F:\PROJECTS\masterthesis\gdrive\masterthesis\notebooks\global_setup.py"
elif os.path.exists("/content/drive_f"):
    _setup = "/content/drive_f/masterthesis/notebooks/global_setup.py"
else:
    _setup = "/mnt/f/PROJECTS/masterthesis/gdrive/masterthesis/notebooks/global_setup.py"
with open(_setup) as f:
    exec(f.read())


/home/alpineobotics/miniconda3/envs/bda/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


BDA GLOBAL SETUP
Started: 2026-04-29 08:31:43
Python: 3.12.12

[1/7] Directory Structure
----------------------------------------------------------------------
  GDrive (G:):       /content/drive_f/masterthesis OK
  GDrive (F:):       /content/drive_f/masterthesis OK
  Local data (G:):   /content/masterthesis_local/data OK
  Data stack (F:):   /mnt/f/PROJECTS/masterthesis/data_stack OK

  TIER_SELECTION: [0, 1, 2]
  CITY_SELECTION: None (tier filter)
  REQUIRE_UNOSAT: False
  CITIES_TO_PROCESS: 21 cities

[2/7] Credentials
----------------------------------------------------------------------
  Copernicus: inf***
  OpenTopography: OK
  Earthdata: marcoheinzen

[3/7] Python Packages
----------------------------------------------------------------------


<string>:564: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.



  Already installed: 23
  Newly installed:   0
  Failed:            0

[4/7] Global Imports & Configuration
----------------------------------------------------------------------
  All imports loaded

[5/7] Processing Config & SNAP
----------------------------------------------------------------------
  GPT: Usage:
  Temporal baseline: 10-24 days
  Wavelength: 0.0555

[6/7] GPU Status
----------------------------------------------------------------------
  CUDA available: NVIDIA GeForce RTX 2070 SUPER
    CUDA version: 12.8

[7/7] Disk Space
----------------------------------------------------------------------
  GDrive (G:)     909.8/7452.0 GB (6542.2 GB free)
  GDrive (F:)     1402.2/3726.0 GB (2323.8 GB free)
  Local data      11557.6/14901.9 GB (3344.3 GB free)
  Data stack      1402.2/3726.0 GB (2323.8 GB free)
  WSL ext4        68.6/1006.9 GB (887.1 GB free)

GLOBAL SETUP COMPLETE
  Torch device: cuda
  Cities: 21, CITY=Avdiivka
  Functions: load_aoi(), load_aoi_gdf(), load_aoi_

# CELL S0: LOAD v2 MANIFEST + BUILDINGS + HELPERS
- Loads ONLY bda_buildings.parquet (metadata). Each D-cell loads its own experiment parquet.

In [3]:
# @title CELL S0: LOAD V3 MANIFEST + POINTS + HELPERS
import sys, importlib, gc
import numpy as np
import pandas as pd
import json as _json
from pathlib import Path

# Central column-role filter: single source of truth for id / label / metadata
# classification. Replaces the hand-maintained _NON_FEATURE set that previously
# lived here -- every item in that set is now covered (verified 47/47) by
# metadata_filter.is_non_feature(), plus the pattern rules catch was_observed_*,
# scenes_observed_*, qa__cloud_freq__*, visibility__*__freq__*, obs_count__*,
# block __count_* which the old set missed.
import metadata_filter
importlib.reload(metadata_filter)
from metadata_filter import is_non_feature
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings("ignore")

print("=" * 70)
print("CELL S0: NB07V3 v1 MANIFEST + POINTS + HELPERS")
print("=" * 70)

# ---- V3 paths ----
DATASET_ROOT_V3 = STACK_DIR / 'dataset' / 'v3'
V3_DIR = DATASET_ROOT_V3
MANIFEST_PATH = V3_DIR / 'parquet_manifest.json'

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"V3 manifest not found: {MANIFEST_PATH}")

with open(MANIFEST_PATH) as f:
    MANIFEST = _json.load(f)
print(f"  Manifest: {len(MANIFEST['parquets'])} parquets")

_tiers = TIER_SELECTION if TIER_SELECTION != "ALL" else [0, 1, 2, 3, 4, 5]

def load_v3_parquet(name):
    dfs = []
    for tier in _tiers:
        p = V3_DIR / f"bda_{name}_t{tier}.parquet"
        if p.exists():
            dfs.append(pd.read_parquet(p))
    if not dfs:
        return None
    return pd.concat(dfs, ignore_index=True)

# ---- points ----
df_points = load_v3_parquet('points')
if df_points is None:
    raise FileNotFoundError("bda_points not found")
if FILTER_UNOSAT_ONLY:
    df_points = df_points[df_points[TARGET_COL].isin([0, 1])].reset_index(drop=True)
print(f"  Points: {len(df_points)} rows, {df_points['city'].nunique()} cities")
print(f"  Damaged: {(df_points[TARGET_COL]==1).sum()}, Undamaged: {(df_points[TARGET_COL]==0).sum()}")

# _NON_FEATURE set removed -- metadata_filter.is_non_feature() is now the
# single source of truth (verified to cover all 43 previously enumerated items
# plus pattern-matched leakage columns).

join_cols = ['point_id', 'city']

def get_analysis_df(pq_name):
    df_pq = load_v3_parquet(pq_name)
    if df_pq is None:
        return None, [], False
    is_w = 'date' not in df_pq.columns
    if is_w:
        bex = [c for c in df_points.columns if c not in df_pq.columns]
        merged = df_pq.merge(df_points[join_cols + bex], on=join_cols, how='left')
    else:
        bex = [c for c in df_points.columns if c not in df_pq.columns and c not in ('date', 'timestep', 'period_label')]
        merged = df_pq.merge(df_points[join_cols + bex], on=join_cols, how='left')
    del df_pq
    if FILTER_UNOSAT_ONLY and TARGET_COL in merged.columns:
        merged = merged[merged[TARGET_COL].isin([0, 1])].reset_index(drop=True)
    if CITY_SELECTION is not None:
        merged = merged[merged['city'].isin(CITY_SELECTION)].reset_index(drop=True)
    feat = [c for c in merged.columns
            if not is_non_feature(c)
            and merged[c].dtype.kind in ('f', 'i', 'u')]
    mb = merged.memory_usage(deep=True).sum() / 1e6
    print(f"  Loaded {pq_name}: {len(merged)} rows, {len(feat)} features, {mb:.1f} MB")
    return merged, feat, is_w

CITIES_TO_PROCESS = sorted(df_points['city'].unique())
print(f"  Cities: {len(CITIES_TO_PROCESS)}")

# ---- shared RF helper ----
def run_rf_cv(X, y, groups=None, label="", n_folds=5):
    imp = SimpleImputer(strategy='median')
    X_imp = imp.fit_transform(X)
    if groups is not None:
        cv = GroupKFold(n_splits=min(n_folds, len(np.unique(groups))))
        splits = list(cv.split(X_imp, y, groups))
    else:
        cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
        splits = list(cv.split(X_imp, y))
    y_proba = np.zeros(len(y))
    fold_id = np.full(len(y), -1, dtype=int)
    for k, (train_idx, test_idx) in enumerate(splits):
        rf = RandomForestClassifier(**RF_PARAMS)
        rf.fit(X_imp[train_idx], y[train_idx])
        y_proba[test_idx] = rf.predict_proba(X_imp[test_idx])[:, 1]
        fold_id[test_idx] = k
    auc = roc_auc_score(y, y_proba)
    y_pred = (y_proba >= 0.5).astype(int)
    f1 = f1_score(y, y_pred, zero_division=0)
    prec = precision_score(y, y_pred, zero_division=0)
    rec = recall_score(y, y_pred, zero_division=0)
    cv_type = "GroupKFold" if groups is not None else "StratifiedKFold"
    print(f"    {label:45s} AUC={auc:.3f} F1={f1:.3f} P={prec:.3f} R={rec:.3f} [{cv_type}]")
    return {'auc': auc, 'f1': f1, 'precision': prec, 'recall': rec,
            'y_proba': y_proba, 'fold_id': fold_id}

# ---- output helpers ----
import matplotlib.pyplot as plt
OUT_DIR = RESULTS_ROOT / 'nb07v3'
OUT_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name, subdir=''):
    d = OUT_DIR / subdir if subdir else OUT_DIR
    d.mkdir(parents=True, exist_ok=True)
    fig.savefig(d / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

def save_result(df_result, name, subdir=''):
    d = OUT_DIR / subdir if subdir else OUT_DIR
    d.mkdir(parents=True, exist_ok=True)
    out = d / f'{name}.csv'
    df_result.to_csv(out, index=False)
    print(f"    saved -> {out.name} ({len(df_result)} rows)")

NB07_RESULTS = []

# ---- NB12 overlap-analysis: canonical OOF predictions schema ----
import hashlib as _hashlib, datetime as _dt
EXPERIMENT_ID = _dt.datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + _hashlib.md5(
    f'nb07_{TIER_SELECTION}_{FILTER_UNOSAT_ONLY}'.encode()).hexdigest()[:6]
print(f"  EXPERIMENT_ID: {EXPERIMENT_ID}")

OOF_DIR = OUT_DIR / 'oof_predictions'
OOF_DIR.mkdir(parents=True, exist_ok=True)

def save_oof(res, model_id, building_ids, cities, y_true,
             variant_id='', is_final=False, threshold=0.5):
    if 'y_proba' not in res or 'fold_id' not in res:
        print(f"    [save_oof] skipped {model_id}: missing y_proba or fold_id")
        return None
    y_proba = np.asarray(res['y_proba'])
    fold = np.asarray(res['fold_id']).astype(int)
    yt = np.asarray(y_true).astype(int)
    yp = (y_proba >= threshold).astype(int)
    cm_class = np.where(yt == 1,
                        np.where(yp == 1, 'TP', 'FN'),
                        np.where(yp == 1, 'FP', 'TN'))
    oof_df = pd.DataFrame({
        'point_id':   np.asarray(building_ids),
        'city':          np.asarray(cities),
        'fold_id':       fold,
        'y_true':        yt,
        'y_proba':       y_proba.astype(float),
        'y_pred':        yp,
        'cm_class':      cm_class,
        'model_id':      model_id,
        'experiment_id': EXPERIMENT_ID,
        'variant_id':    variant_id,
        'is_final':      bool(is_final),
    })
    out = OOF_DIR / f'oof_{model_id}__{EXPERIMENT_ID}.parquet'
    oof_df.to_parquet(out, index=False)
    n_tp = int((cm_class == 'TP').sum()); n_fn = int((cm_class == 'FN').sum())
    n_fp = int((cm_class == 'FP').sum()); n_tn = int((cm_class == 'TN').sum())
    print(f"    saved oof -> {out.name}  TP={n_tp} FN={n_fn} FP={n_fp} TN={n_tn}")
    return out

def log_experiment(name, parquet, cv_method, auc, f1, n_features, n_cities, n_buildings, extra=None):
    row = {'experiment': name, 'parquet': parquet, 'cv_method': cv_method,
           'auc': auc, 'f1': f1, 'n_features': n_features, 'n_cities': n_cities,
           'n_buildings': n_buildings}
    if extra:
        row.update(extra)
    NB07_RESULTS.append(row)


CELL S0: NB07V3 v1 MANIFEST + POINTS + HELPERS
  Manifest: 36 parquets
  Points: 63243 rows, 21 cities
  Damaged: 8247, Undamaged: 54996
  Cities: 21
  EXPERIMENT_ID: 20260429_083322_06ea54


In [4]:
# @title CELL S0b: OOF PLOT + SUMMARY HELPERS
# TP=red (destroyed, correctly predicted)
# TN=green (not destroyed, correctly predicted)
# FP=pink (predicted damaged but undamaged)
# FN=gold (actually damaged but missed)

CM_COLORS = {'TP': '#d62728', 'TN': '#2ca02c', 'FP': '#ff9ecb', 'FN': '#ffd700'}
CM_LABELS = {'TP': 'Destroyed (TP)', 'TN': 'Not destroyed (TN)',
             'FP': 'False positive (FP)', 'FN': 'False negative (FN)'}

def print_cm_summary(oof_path_or_df):
    if isinstance(oof_path_or_df, (str, Path)):
        oof = pd.read_parquet(oof_path_or_df)
    else:
        oof = oof_path_or_df
    model_id = oof['model_id'].iloc[0] if 'model_id' in oof.columns else '(unknown)'
    print(f"  model_id: {model_id}   n={len(oof)}")
    overall = oof['cm_class'].value_counts()
    for cls in ['TP', 'TN', 'FP', 'FN']:
        n = int(overall.get(cls, 0))
        pct = 100.0 * n / len(oof) if len(oof) else 0
        print(f"    {cls:3s} {CM_LABELS[cls]:28s}  n={n:6d}  ({pct:5.1f}%)")
    by_city = (oof.groupby('city')['cm_class']
                 .value_counts().unstack(fill_value=0))
    for cls in ['TP', 'TN', 'FP', 'FN']:
        if cls not in by_city.columns:
            by_city[cls] = 0
    by_city = by_city[['TP', 'TN', 'FP', 'FN']]
    by_city['recall']    = by_city['TP'] / (by_city['TP'] + by_city['FN']).replace(0, np.nan)
    by_city['precision'] = by_city['TP'] / (by_city['TP'] + by_city['FP']).replace(0, np.nan)
    print(f"\n  Per-city:")
    print(by_city.to_string(float_format=lambda x: f'{x:.3f}' if pd.notna(x) else '-'))
    return by_city

def plot_cm_spatial(oof_path_or_df, save_name=None, figsize=(14, 10),
                    buildings_df=None, point_size=4):
    if isinstance(oof_path_or_df, (str, Path)):
        oof = pd.read_parquet(oof_path_or_df)
    else:
        oof = oof_path_or_df
    bdf = buildings_df if buildings_df is not None else df_points
    # V3: use x_utm/y_utm as primary coordinates; fall back to centroid_x/y (V2)
    if 'x_utm' in bdf.columns and 'y_utm' in bdf.columns:
        _cx, _cy = 'x_utm', 'y_utm'
    elif 'centroid_x' in bdf.columns and 'centroid_y' in bdf.columns:
        _cx, _cy = 'centroid_x', 'centroid_y'
    else:
        print("  plot_cm_spatial: no coordinate columns (x_utm/centroid_x)")
        return None
    plot_df = oof.merge(bdf[['point_id', 'city', _cx, _cy]],
                        on=['point_id', 'city'], how='left')
    plot_df = plot_df.dropna(subset=[_cx, _cy])
    cities = sorted(plot_df['city'].unique())
    if not cities:
        return None
    ncols = min(3, len(cities))
    nrows = (len(cities) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)
    model_id = oof['model_id'].iloc[0] if 'model_id' in oof.columns else ''
    fig.suptitle(f'Confusion-matrix map: {model_id}', fontsize=11)
    # fixed draw order so reds/golds sit on top of greens/pinks
    draw_order = ['TN', 'FP', 'FN', 'TP']
    for i, city in enumerate(cities):
        ax = axes[i // ncols][i % ncols]
        cdf = plot_df[plot_df['city'] == city]
        for cls in draw_order:
            pts = cdf[cdf['cm_class'] == cls]
            if len(pts) == 0:
                continue
            ax.scatter(pts[_cx], pts[_cy],
                       c=CM_COLORS[cls], s=point_size, alpha=0.75,
                       edgecolors='none',
                       label=f"{CM_LABELS[cls]} (n={len(pts)})")
        ax.set_title(f"{city}  (n={len(cdf)})", fontsize=9)
        ax.set_aspect('equal')
        ax.legend(loc='best', fontsize=6, markerscale=2, framealpha=0.85)
        ax.tick_params(labelsize=7)
    for j in range(len(cities), nrows * ncols):
        axes[j // ncols][j % ncols].axis('off')
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    if save_name:
        save_fig(fig, save_name, 'cm_plots')
    return fig

def plot_cm_oof_all(oof_dir=None, skip_variants=('leakage_demo_with_leaks', 'leakage_demo_clean')):
    d = Path(oof_dir) if oof_dir else OOF_DIR
    files = sorted(d.glob('oof_*.parquet'))
    print(f"  Found {len(files)} OOF parquets in {d}")
    for p in files:
        oof = pd.read_parquet(p)
        v = oof['variant_id'].iloc[0] if 'variant_id' in oof.columns else ''
        if v in skip_variants:
            print(f"  skip {p.name}  (variant_id={v})")
            continue
        print(f"\n  {p.name}")
        print_cm_summary(oof)
        plot_cm_spatial(oof, save_name=p.stem)


# CELL D1: PRE vs POST SAR BASELINES (prepost_single_card A13, quick win) per city

In [5]:
# @title CELL D1: PRE vs POST SAR BASELINES (prepost_single_card A13, per city)
import gc
from datetime import datetime as _dt

print("=" * 70)
print("CELL D1: PRE vs POST SAR BASELINES (prepost_single_card)")
print("  pre_vv/vh = pre-battle CARD, post_vv/vh = post-battle CARD")
print("=" * 70)

df_a13, feat_a13, _ = get_analysis_df('prepost_single_card')
feat_a13 = exclude_leakage(feat_a13)

pre_cols = [c for c in feat_a13 if c.startswith('pre_')]
post_cols = [c for c in feat_a13 if c.startswith('post_')]
delta_cols = [c for c in feat_a13 if c.startswith('delta_')]

nan_pct = df_a13[pre_cols + post_cols + delta_cols].isna().mean()
pre_cols = [c for c in pre_cols if nan_pct[c] < 1.0]
post_cols = [c for c in post_cols if nan_pct[c] < 1.0]
delta_cols = [c for c in delta_cols if nan_pct[c] < 1.0]

print(f"  Pre: {pre_cols}")
print(f"  Post: {post_cols}")
print(f"  Delta: {delta_cols}")

sets = {
    'pre_only': pre_cols,
    'post_only': post_cols,
    'delta_only': delta_cols,
    'pre+post': pre_cols + post_cols,
    'pre+post+delta': pre_cols + post_cols + delta_cols,
}

print(f"\n  Per-city StratifiedKFold:")
print(f"  {'City':<18s} {'N':>7s} {'Dam':>5s} {'Ratio':>6s} {'Days':>5s}", end="")
for label in sets:
    print(f" {label:>13s}", end="")
print(f" {'d-p':>6s} {'p>.5':>5s}")

for city in sorted(df_a13['city'].unique()):
    cdf = df_a13[df_a13['city'] == city]
    n_dam = (cdf[TARGET_COL] == 1).sum()
    if n_dam < 10 or (len(cdf) - n_dam) < 10:
        continue

    # damage ratio
    ratio = n_dam / len(cdf)

    # battle days
    battle_days = 0
    if 'battle_start' in cdf.columns and 'battle_stop' in cdf.columns:
        bs = cdf['battle_start'].iloc[0]
        be = cdf['battle_stop'].iloc[0]
        if pd.notna(bs) and pd.notna(be):
            try:
                battle_days = (pd.Timestamp(be) - pd.Timestamp(bs)).days
            except Exception:
                battle_days = 0

    y = cdf[TARGET_COL].values
    aucs = {}
    for label, cols in sets.items():
        if len(cols) < 2:
            aucs[label] = None
            continue
        X = cdf[cols].values
        imp = SimpleImputer(strategy='median')
        X_imp = imp.fit_transform(X)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        y_proba = np.zeros(len(y))
        for train_idx, test_idx in cv.split(X_imp, y):
            rf = RandomForestClassifier(**RF_PARAMS)
            rf.fit(X_imp[train_idx], y[train_idx])
            y_proba[test_idx] = rf.predict_proba(X_imp[test_idx])[:, 1]
        aucs[label] = roc_auc_score(y, y_proba)

    # derived columns
    auc_pre = aucs.get('pre_only')
    auc_delta = aucs.get('delta_only')
    d_minus_p = (auc_delta - auc_pre) if auc_delta is not None and auc_pre is not None else None
    pre_gt_05 = (auc_pre > 0.5) if auc_pre is not None else None

    print(f"  {city:<18s} {len(cdf):>7d} {n_dam:>5d} {ratio:>5.1%} {battle_days:>5d}", end="")
    for label in sets:
        v = aucs[label]
        print(f" {v:>13.3f}" if v is not None else f" {'SKIP':>13s}", end="")
    print(f" {d_minus_p:>+6.3f}" if d_minus_p is not None else f" {'?':>6s}", end="")
    print(f" {'YES' if pre_gt_05 else 'no':>5s}")

print(f"\n  Dietrich paper: AUC={DIETRICH_PAPER['auc']:.3f}")
print(f"  d-p = delta_AUC - pre_AUC: positive = change detection adds signal")
print(f"  p>.5 = pre-battle AUC > 0.5: SAR encodes building structure before damage")

del df_a13; gc.collect()
print("  Memory freed")

CELL D1: PRE vs POST SAR BASELINES (prepost_single_card)
  pre_vv/vh = pre-battle CARD, post_vv/vh = post-battle CARD
  Loaded prepost_single_card: 63243 rows, 6 features, 16.7 MB
  Pre: ['pre_vv', 'pre_vh']
  Post: ['post_vv', 'post_vh']
  Delta: ['delta_vv', 'delta_vh']

  Per-city StratifiedKFold:
  City                     N   Dam  Ratio  Days      pre_only     post_only    delta_only      pre+post pre+post+delta    d-p  p>.5
  Avdiivka              1186    93  7.8%   723         0.517         0.591         0.552         0.684         0.722 +0.035   YES
  Borodyanka             507    68 13.4%    32         0.480         0.509         0.565         0.504         0.556 +0.085    no
  Bucha                 1662   194 11.7%    32         0.571         0.518         0.476         0.563         0.536 -0.095   YES
  Chernihiv             2810   358 12.7%    39         0.516         0.486         0.514         0.530         0.531 -0.001   YES
  Chornobaivka           142    13  9.2%   260

In [6]:
# @title CELL D1x: BUILDING SIZE vs DAMAGE DETECTABILITY
import gc

print("=" * 70)
print("CELL D1x: BUILDING SIZE vs DAMAGE DETECTABILITY")
print("=" * 70)

df_a13, feat_a13, _ = get_analysis_df('prepost_single_card')
feat_a13 = exclude_leakage(feat_a13)

pre_cols = [c for c in feat_a13 if c.startswith('pre_')]
post_cols = [c for c in feat_a13 if c.startswith('post_')]
delta_cols = [c for c in feat_a13 if c.startswith('delta_')]

print(f"\n  {'City':<18s} {'N':>6s} {'Dam':>5s} {'Ratio':>5s}"
      f" {'AreaMed':>8s} {'DamMed':>8s} {'UndMed':>8s} {'D/U':>6s}"
      f" {'nPxDam':>6s} {'nPxUnd':>6s}"
      f" {'PreAUC':>7s} {'DltAUC':>7s}")

city_rows = []

for city in sorted(df_a13['city'].unique()):
    cdf = df_a13[df_a13['city'] == city]
    n_dam = (cdf[TARGET_COL] == 1).sum()
    if n_dam < 10 or (len(cdf) - n_dam) < 10:
        continue

    dam = cdf[cdf[TARGET_COL] == 1]
    und = cdf[cdf[TARGET_COL] == 0]

    area_damaged_median = dam['area_m2'].median() if 'area_m2' in dam.columns else np.nan
    area_undamaged_median = und['area_m2'].median() if 'area_m2' in und.columns else np.nan
    area_all_median = cdf['area_m2'].median() if 'area_m2' in cdf.columns else np.nan
    area_ratio_d_vs_u = area_damaged_median / area_undamaged_median if area_undamaged_median > 0 else np.nan

    npx_dam = dam['n_pixels'].median() if 'n_pixels' in dam.columns else np.nan
    npx_und = und['n_pixels'].median() if 'n_pixels' in und.columns else np.nan

    y = cdf[TARGET_COL].values
    auc_pre = np.nan
    auc_delta = np.nan
    imp = SimpleImputer(strategy='median')
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    for label, cols, target in [('pre', pre_cols, 'auc_pre'), ('delta', delta_cols, 'auc_delta')]:
        if len(cols) < 2:
            continue
        X = imp.fit_transform(cdf[cols].values)
        y_proba = np.zeros(len(y))
        for train_idx, test_idx in cv.split(X, y):
            rf = RandomForestClassifier(**RF_PARAMS)
            rf.fit(X[train_idx], y[train_idx])
            y_proba[test_idx] = rf.predict_proba(X[test_idx])[:, 1]
        if target == 'auc_pre':
            auc_pre = roc_auc_score(y, y_proba)
        else:
            auc_delta = roc_auc_score(y, y_proba)

    ratio = n_dam / len(cdf)
    print(f"  {city:<18s} {len(cdf):>6d} {n_dam:>5d} {ratio:>4.1%}"
          f" {area_all_median:>8.0f} {area_damaged_median:>8.0f} {area_undamaged_median:>8.0f} {area_ratio_d_vs_u:>6.2f}"
          f" {npx_dam:>6.0f} {npx_und:>6.0f}"
          f" {auc_pre:>7.3f} {auc_delta:>7.3f}")

    city_rows.append({
        'city': city, 'n': len(cdf), 'n_dam': n_dam, 'ratio': ratio,
        'area_all_median': area_all_median,
        'area_damaged_median': area_damaged_median,
        'area_undamaged_median': area_undamaged_median,
        'area_ratio_d_vs_u': area_ratio_d_vs_u,
        'npx_dam': npx_dam, 'npx_und': npx_und,
        'auc_pre': auc_pre, 'auc_delta': auc_delta,
    })

if city_rows:
    _df = pd.DataFrame(city_rows)
    from scipy.stats import spearmanr

    print(f"\n  CORRELATIONS (Spearman, across {len(_df)} cities):")
    for col in ['area_damaged_median', 'area_ratio_d_vs_u', 'npx_dam', 'ratio']:
        for auc_col in ['auc_pre', 'auc_delta']:
            valid = _df[[col, auc_col]].dropna()
            if len(valid) >= 5:
                r, p = spearmanr(valid[col], valid[auc_col])
                sig = '*' if p < 0.05 else ' '
                print(f"    {col:25s} vs {auc_col:10s}: r={r:+.3f} p={p:.3f} {sig}")

    print(f"\n  INTERPRETATION:")
    # V3 guard: area_m2 / n_pixels columns do not exist on per-point parquets
    # (no building footprint -> no footprint area). Skip area-based interpretation.
    _area_valid = _df['area_damaged_median'].dropna()
    _auc_valid = _df['auc_pre'].dropna()
    if len(_area_valid) >= 5 and len(_area_valid) == len(_auc_valid):
        r_area, _ = spearmanr(_area_valid, _auc_valid)
        r_ratio, _ = spearmanr(_df['area_ratio_d_vs_u'].dropna(), _auc_valid)
        if r_area > 0.3:
            print(f"    area_damaged_median vs auc_pre: r={r_area:+.3f}")
            print(f"    -> LARGER damaged buildings = higher pre-AUC")
            print(f"    -> SAR at 10m resolves large buildings, pre-battle signal is STRUCTURAL SIZE")
            print(f"    -> operational BDA needs VHR for small-building cities")
        else:
            print(f"    area_damaged_median vs auc_pre: r={r_area:+.3f} -> no strong size-AUC link")

        if abs(r_ratio) > 0.3:
            print(f"    area_ratio_d_vs_u vs auc_pre: r={r_ratio:+.3f}")
            print(f"    -> damaged buildings {'larger' if r_ratio > 0 else 'smaller'} relative to undamaged")
            print(f"    -> UNOSAT labels biased toward {'large' if r_ratio > 0 else 'small'} structures")
    else:
        print(f"    area_m2 not available (V3 per-point sample unit has no footprint area)")
        print(f"    -> building-size vs detectability analysis not applicable for V3")

    save_result(_df, 'building_size_vs_auc', 'cell_d1x')

del df_a13; gc.collect()
print("  Memory freed")

CELL D1x: BUILDING SIZE vs DAMAGE DETECTABILITY
  Loaded prepost_single_card: 63243 rows, 6 features, 16.7 MB

  City                    N   Dam Ratio  AreaMed   DamMed   UndMed    D/U nPxDam nPxUnd  PreAUC  DltAUC
  Avdiivka             1186    93 7.8%      nan      nan      nan    nan    nan    nan   0.517   0.552
  Borodyanka            507    68 13.4%      nan      nan      nan    nan    nan    nan   0.480   0.565
  Bucha                1662   194 11.7%      nan      nan      nan    nan    nan    nan   0.571   0.476
  Chernihiv            2810   358 12.7%      nan      nan      nan    nan    nan    nan   0.516   0.514
  Chornobaivka          142    13 9.2%      nan      nan      nan    nan    nan    nan   0.581   0.827
  Dmytrivka            3136   361 11.5%      nan      nan      nan    nan    nan    nan   0.632   0.645
  Hostomel             4497   618 13.7%      nan      nan      nan    nan    nan    nan   0.498   0.525
  Irpin                2749   308 11.2%      nan      nan  

In [7]:
# @title CELL D1y: PRE vs POST MS BASELINES (composite_prepost_bands A9, per city)
import gc

print("=" * 70)
print("CELL D1y: PRE vs POST MS BASELINES (composite_prepost_bands)")
print("  pre = prebattle/winter composites, post = post_winter composites")
print("=" * 70)

df_a9, feat_a9, _ = get_analysis_df('composite_prepost_bands')
feat_a9 = exclude_leakage(feat_a9)

pre_cols = [c for c in feat_a9 if 'prebattle_baseline' in c or 'winter_baseline' in c]
post_cols = [c for c in feat_a9 if 'post_winter_baseline' in c]

pre_cols = list(dict.fromkeys(pre_cols))
post_cols = list(dict.fromkeys(post_cols))
all_check = list(dict.fromkeys(pre_cols + post_cols))
nan_pct = df_a9[all_check].isna().mean()
pre_cols = [c for c in pre_cols if nan_pct[c] < 1.0]
post_cols = [c for c in post_cols if nan_pct[c] < 1.0]

print(f"  Pre (prebattle + winter baseline): {len(pre_cols)} features")
print(f"  Post (post_winter_baseline): {len(post_cols)} features")

print(f"\n  {'City':<18s} {'N':>6s} {'Dam':>5s} {'Ratio':>5s}"
      f" {'AreaMed':>8s} {'DamMed':>8s} {'UndMed':>8s} {'D/U':>6s}"
      f" {'PreAUC':>7s} {'PostAUC':>7s} {'BothAUC':>8s} {'P-Pre':>6s} {'Pre>.5':>6s}")

city_rows = []

for city in sorted(df_a9['city'].unique()):
    cdf = df_a9[df_a9['city'] == city]
    n_dam = (cdf[TARGET_COL] == 1).sum()
    if n_dam < 10 or (len(cdf) - n_dam) < 10:
        continue

    dam = cdf[cdf[TARGET_COL] == 1]
    und = cdf[cdf[TARGET_COL] == 0]

    area_damaged_median = dam['area_m2'].median() if 'area_m2' in dam.columns else np.nan
    area_undamaged_median = und['area_m2'].median() if 'area_m2' in und.columns else np.nan
    area_all_median = cdf['area_m2'].median() if 'area_m2' in cdf.columns else np.nan
    area_ratio_d_vs_u = area_damaged_median / area_undamaged_median if area_undamaged_median > 0 else np.nan

    y = cdf[TARGET_COL].values
    imp = SimpleImputer(strategy='median')
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    aucs = {}
    for label, cols in [('pre', pre_cols), ('post', post_cols), ('both', list(dict.fromkeys(pre_cols + post_cols)))]:
        if len(cols) < 2:
            aucs[label] = np.nan
            continue
        X = imp.fit_transform(cdf[cols].values)
        y_proba = np.zeros(len(y))
        for train_idx, test_idx in cv.split(X, y):
            rf = RandomForestClassifier(**RF_PARAMS)
            rf.fit(X[train_idx], y[train_idx])
            y_proba[test_idx] = rf.predict_proba(X[test_idx])[:, 1]
        aucs[label] = roc_auc_score(y, y_proba)

    ratio = n_dam / len(cdf)
    post_minus_pre = aucs['post'] - aucs['pre'] if np.isfinite(aucs['post']) and np.isfinite(aucs['pre']) else np.nan
    pre_gt_05 = aucs['pre'] > 0.5 if np.isfinite(aucs['pre']) else False

    print(f"  {city:<18s} {len(cdf):>6d} {n_dam:>5d} {ratio:>4.1%}"
          f" {area_all_median:>8.0f} {area_damaged_median:>8.0f} {area_undamaged_median:>8.0f} {area_ratio_d_vs_u:>6.2f}"
          f" {aucs['pre']:>7.3f} {aucs['post']:>7.3f} {aucs['both']:>8.3f}"
          f" {post_minus_pre:>+6.3f} {'YES' if pre_gt_05 else 'no':>6s}")

    city_rows.append({
        'city': city, 'n': len(cdf), 'n_dam': n_dam, 'ratio': ratio,
        'area_all_median': area_all_median,
        'area_damaged_median': area_damaged_median,
        'area_undamaged_median': area_undamaged_median,
        'area_ratio_d_vs_u': area_ratio_d_vs_u,
        'auc_pre': aucs['pre'], 'auc_post': aucs['post'], 'auc_both': aucs['both'],
        'post_minus_pre': post_minus_pre,
    })

if city_rows:
    _df = pd.DataFrame(city_rows)
    from scipy.stats import spearmanr

    print(f"\n  CORRELATIONS (Spearman, across {len(_df)} cities):")
    for col in ['area_damaged_median', 'area_ratio_d_vs_u', 'ratio']:
        for auc_col in ['auc_pre', 'auc_post', 'post_minus_pre']:
            valid = _df[[col, auc_col]].dropna()
            if len(valid) >= 5:
                r, p = spearmanr(valid[col], valid[auc_col])
                sig = '*' if p < 0.05 else ' '
                print(f"    {col:25s} vs {auc_col:15s}: r={r:+.3f} p={p:.3f} {sig}")

    print(f"\n  INTERPRETATION:")
    mean_pre = _df['auc_pre'].mean()
    mean_post = _df['auc_post'].mean()
    mean_delta = _df['post_minus_pre'].mean()
    print(f"    Mean pre-AUC:  {mean_pre:.3f}")
    print(f"    Mean post-AUC: {mean_post:.3f}")
    print(f"    Mean P-Pre:    {mean_delta:+.3f}")
    if mean_delta > 0.03:
        print(f"    -> Post-battle MS composites carry MORE damage signal than pre-battle")
        print(f"    -> Damage changes optical signature (rubble, vegetation loss, burn scars)")
    elif mean_delta < -0.03:
        print(f"    -> Pre-battle MS carries MORE signal (unexpected, check composite windows)")
    else:
        print(f"    -> Pre and post MS carry similar signal")

    print(f"\n  COMPARE WITH SAR (D1):")
    print(f"    MS pre-AUC mean:  {mean_pre:.3f}")
    print(f"    MS post-AUC mean: {mean_post:.3f}")
    print(f"    (SAR values from D1 cell for comparison)")

    save_result(_df, 'ms_prepost_vs_auc', 'cell_d1y')

del df_a9; gc.collect()
print("  Memory freed")

CELL D1y: PRE vs POST MS BASELINES (composite_prepost_bands)
  pre = prebattle/winter composites, post = post_winter composites
  Loaded composite_prepost_bands: 62043 rows, 79 features, 32.5 MB
  Pre (prebattle + winter baseline): 45 features
  Post (post_winter_baseline): 15 features

  City                    N   Dam Ratio  AreaMed   DamMed   UndMed    D/U  PreAUC PostAUC  BothAUC  P-Pre Pre>.5
  Avdiivka             1186    93 7.8%      nan      nan      nan    nan   0.797   0.716    0.797 -0.081    YES
  Bucha                1662   194 11.7%      nan      nan      nan    nan   0.699   0.704    0.699 +0.005    YES
  Chernihiv            2810   358 12.7%      nan      nan      nan    nan   0.663   0.546    0.663 -0.116    YES
  Chornobaivka          142    13 9.2%      nan      nan      nan    nan   0.625   0.531    0.625 -0.094    YES
  Dmytrivka            3136   361 11.5%      nan      nan      nan    nan   0.572   0.499    0.572 -0.073    YES
  Hostomel             4497   618 13

In [8]:
# @title CELL D1: PRE vs POST SAR BASELINES (prepost_single_card A13, quick win)
# Goal: Use pre-battle and post-battle CARD single-scene baselines.
# This is the simplest possible test: does post-battle SAR differ from pre-battle?
import gc

print("=" * 70)
print("CELL D1: PRE vs POST SAR BASELINES (prepost_single_card)")
print("  pre_vv/vh = pre-battle CARD, post_vv/vh = post-battle CARD")
print("=" * 70)

df_a13, feat_a13, _ = get_analysis_df('prepost_single_card')
feat_a13 = exclude_leakage(feat_a13)

# separate pre and post features
pre_cols = [c for c in feat_a13 if c.startswith('pre_')]
post_cols = [c for c in feat_a13 if c.startswith('post_')]
delta_cols = [c for c in feat_a13 if c.startswith('delta_')]

# drop all-NaN
nan_pct = df_a13[pre_cols + post_cols + delta_cols].isna().mean()
pre_cols = [c for c in pre_cols if nan_pct[c] < 1.0]
post_cols = [c for c in post_cols if nan_pct[c] < 1.0]
delta_cols = [c for c in delta_cols if nan_pct[c] < 1.0]

print(f"  Pre-battle: {pre_cols}")
print(f"  Post-battle: {post_cols}")
print(f"  Delta: {delta_cols}")

groups = df_a13['city'].values
y = df_a13[TARGET_COL].values

# test feature sets
sets = {
    'pre_only': pre_cols,
    'post_only': post_cols,
    'delta_only': delta_cols,
    'pre+post': pre_cols + post_cols,
    'pre+post+delta': pre_cols + post_cols + delta_cols,
}

print(f"\n  GroupKFold across {df_a13['city'].nunique()} cities:")
for label, cols in sets.items():
    if len(cols) < 2:
        print(f"    {label:30s} SKIP ({len(cols)} features)")
        continue
    res = run_rf_cv(df_a13[cols].values, y, groups=groups,
                    label=f"{label} ({len(cols)} feat)")
    log_experiment(f'D1_{label}', 'prepost_single_card', 'GroupKFold',
                   res['auc'], res['f1'], len(cols),
                   df_a13['city'].nunique(), len(df_a13))
    save_oof(res, f'D1_{label}',
             df_a13['point_id'].values, df_a13['city'].values, y,
             variant_id=f'feat_set={label}')

CELL D1: PRE vs POST SAR BASELINES (prepost_single_card)
  pre_vv/vh = pre-battle CARD, post_vv/vh = post-battle CARD
  Loaded prepost_single_card: 63243 rows, 6 features, 16.7 MB
  Pre-battle: ['pre_vv', 'pre_vh']
  Post-battle: ['post_vv', 'post_vh']
  Delta: ['delta_vv', 'delta_vh']

  GroupKFold across 21 cities:
    pre_only (2 feat)                             AUC=0.505 F1=0.030 P=0.138 R=0.017 [GroupKFold]
    saved oof -> oof_D1_pre_only__20260429_083322_06ea54.parquet  TP=138 FN=8109 FP=865 TN=54131
    post_only (2 feat)                            AUC=0.506 F1=0.027 P=0.126 R=0.015 [GroupKFold]
    saved oof -> oof_D1_post_only__20260429_083322_06ea54.parquet  TP=124 FN=8123 FP=862 TN=54134
    delta_only (2 feat)                           AUC=0.512 F1=0.028 P=0.137 R=0.016 [GroupKFold]
    saved oof -> oof_D1_delta_only__20260429_083322_06ea54.parquet  TP=129 FN=8118 FP=810 TN=54186
    pre+post (4 feat)                             AUC=0.503 F1=0.005 P=0.200 R=0.002 [GroupKF

In [9]:

# @title CELL D1: PRE vs POST SAR BASELINES (prepost_single_card A13, per city)
import gc

print("=" * 70)
print("CELL D1: PRE vs POST SAR BASELINES (prepost_single_card)")
print("  pre_vv/vh = pre-battle CARD, post_vv/vh = post-battle CARD")
print("=" * 70)

df_a13, feat_a13, _ = get_analysis_df('prepost_single_card')
feat_a13 = exclude_leakage(feat_a13)

pre_cols = [c for c in feat_a13 if c.startswith('pre_')]
post_cols = [c for c in feat_a13 if c.startswith('post_')]
delta_cols = [c for c in feat_a13 if c.startswith('delta_')]

nan_pct = df_a13[pre_cols + post_cols + delta_cols].isna().mean()
pre_cols = [c for c in pre_cols if nan_pct[c] < 1.0]
post_cols = [c for c in post_cols if nan_pct[c] < 1.0]
delta_cols = [c for c in delta_cols if nan_pct[c] < 1.0]

print(f"  Pre: {pre_cols}")
print(f"  Post: {post_cols}")
print(f"  Delta: {delta_cols}")

sets = {
    'pre_only': pre_cols,
    'post_only': post_cols,
    'delta_only': delta_cols,
    'pre+post': pre_cols + post_cols,
    'pre+post+delta': pre_cols + post_cols + delta_cols,
}

print(f"\n  Per-city StratifiedKFold:")
print(f"  {'City':<22s} {'N':>7s} {'Dam':>5s}", end="")
for label in sets:
    print(f" {label:>16s}", end="")
print()

for city in sorted(df_a13['city'].unique()):
    cdf = df_a13[df_a13['city'] == city]
    n_dam = (cdf[TARGET_COL] == 1).sum()
    if n_dam < 10 or (len(cdf) - n_dam) < 10:
        continue
    y = cdf[TARGET_COL].values
    print(f"  {city:<22s} {len(cdf):>7d} {n_dam:>5d}", end="")
    for label, cols in sets.items():
        if len(cols) < 2:
            print(f" {'SKIP':>16s}", end="")
            continue
        X = cdf[cols].values
        imp = SimpleImputer(strategy='median')
        X_imp = imp.fit_transform(X)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        y_proba = np.zeros(len(y))
        for train_idx, test_idx in cv.split(X_imp, y):
            rf = RandomForestClassifier(**RF_PARAMS)
            rf.fit(X_imp[train_idx], y[train_idx])
            y_proba[test_idx] = rf.predict_proba(X_imp[test_idx])[:, 1]
        auc = roc_auc_score(y, y_proba)
        print(f" {auc:>16.3f}", end="")
    print()

print(f"\n  Dietrich paper: AUC={DIETRICH_PAPER['auc']:.3f}")
print(f"  If delta > pre: change detection adds damage signal")
print(f"  If pre > 0.5: pre-battle SAR encodes building structure")

del df_a13; gc.collect()
print("  Memory freed")
print(f"\n  Dietrich paper: AUC={DIETRICH_PAPER['auc']:.3f} (all blocks, dynamic labeling, random CV)")
print(f"  If delta > pre: change detection adds real damage signal")
print(f"  If pre > 0.5: pre-battle SAR encodes building structure (size, material)")

CELL D1: PRE vs POST SAR BASELINES (prepost_single_card)
  pre_vv/vh = pre-battle CARD, post_vv/vh = post-battle CARD
  Loaded prepost_single_card: 63243 rows, 6 features, 16.7 MB
  Pre: ['pre_vv', 'pre_vh']
  Post: ['post_vv', 'post_vh']
  Delta: ['delta_vv', 'delta_vh']

  Per-city StratifiedKFold:
  City                         N   Dam         pre_only        post_only       delta_only         pre+post   pre+post+delta
  Avdiivka                  1186    93            0.517            0.591            0.552            0.684            0.722
  Borodyanka                 507    68            0.480            0.509            0.565            0.504            0.556
  Bucha                     1662   194            0.571            0.518            0.476            0.563            0.536
  Chernihiv                 2810   358            0.516            0.486            0.514            0.530            0.531
  Chornobaivka               142    13            0.581            0.676      

# CELL D1: DIETRICH-28 PER-CITY DIAGNOSTIC (v2 parquets)

In [10]:
# @title CELL D1: PRE vs first cross block (block_stats, quick win)
# Goal: Use baseline (pre-battle) and blk_pre01 (first post-battle block) features.
# This is the simplest possible test: does post-battle SAR differ from pre-battle?
# NOT a Dietrich replication (that needs all blocks with dynamic labeling).
import gc

print("=" * 70)
print("CELL D1: PRE vs POST SAR BASELINES (block_stats)")
print("  baseline = pre-battle period, blk_pre01 = first post-battle block")
print("=" * 70)

df_blk, feat_blk, _ = get_analysis_df('block_stats')
feat_blk = exclude_leakage(feat_blk)

# separate pre and post features
pre_cols = [c for c in feat_blk if '__baseline__' in c and c.startswith('s1__v')]
post_cols = [c for c in feat_blk if '__blk_pre01__' in c and c.startswith('s1__v')]

# if blk_pre01 doesn't exist, try blk01
if not post_cols:
    post_cols = [c for c in feat_blk if '__blk01__' in c and c.startswith('s1__v')]
if not post_cols:
    # try any blk that isn't baseline
    all_blk = [c for c in feat_blk if c.startswith('s1__v') and '__baseline__' not in c]
    # get unique block names
    blocks = sorted(set(c.split('__')[2] for c in all_blk if len(c.split('__')) >= 4))
    print(f"  Available blocks: {blocks[:10]}")
    if blocks:
        post_cols = [c for c in all_blk if f'__{blocks[0]}__' in c]
        print(f"  Using first non-baseline block: {blocks[0]} ({len(post_cols)} features)")

# drop all-NaN
nan_pct = df_blk[pre_cols + post_cols].isna().mean()
pre_cols = [c for c in pre_cols if nan_pct[c] < 1.0]
post_cols = [c for c in post_cols if nan_pct[c] < 1.0]

# filter to _mean zonal only for Dietrich-like simplicity
pre_mean = [c for c in pre_cols if c.endswith('_mean')]
post_mean = [c for c in post_cols if c.endswith('_mean')]

print(f"  Pre-battle (baseline): {len(pre_cols)} total, {len(pre_mean)} _mean zonal")
print(f"  Post-battle: {len(post_cols)} total, {len(post_mean)} _mean zonal")
if pre_cols:
    print(f"  Pre sample: {pre_cols[:3]}")
if post_cols:
    print(f"  Post sample: {post_cols[:3]}")

groups = df_blk['city'].values
y = df_blk[TARGET_COL].values

# test feature sets
sets = {
    'pre_baseline_mean': pre_mean,
    'post_block_mean': post_mean,
    'pre_baseline_all': pre_cols,
    'post_block_all': post_cols,
    'pre+post_mean': pre_mean + post_mean,
    'pre+post_all': pre_cols + post_cols,
}

print(f"\n  GroupKFold across {df_blk['city'].nunique()} cities:")
for label, cols in sets.items():
    if len(cols) < 2:
        print(f"    {label:30s} SKIP ({len(cols)} features)")
        continue
    res = run_rf_cv(df_blk[cols].values, y, groups=groups,
                    label=f"{label} ({len(cols)} feat)")
    log_experiment(f'D1_{label}', 'block_stats', 'GroupKFold',
                   res['auc'], res['f1'], len(cols),
                   df_blk['city'].nunique(), len(df_blk))
    save_oof(res, f'D1_blk_{label}',
             df_blk['point_id'].values, df_blk['city'].values, y,
             variant_id=f'feat_set={label}')

print(f"\n  Dietrich paper: AUC={DIETRICH_PAPER['auc']:.3f} (all blocks, dynamic labeling, random CV)")

del df_blk; gc.collect()
print("  Memory freed")


CELL D1: PRE vs POST SAR BASELINES (block_stats)
  baseline = pre-battle period, blk_pre01 = first post-battle block
  Loaded block_stats: 63243 rows, 330 features, 96.6 MB
  Pre-battle (baseline): 16 total, 2 _mean zonal
  Post-battle: 22 total, 2 _mean zonal
  Pre sample: ['s1__vh__baseline__count', 's1__vh__baseline__kurtosis', 's1__vh__baseline__max']
  Post sample: ['s1__vh__blk_pre01__count', 's1__vh__blk_pre01__kurtosis', 's1__vh__blk_pre01__max']

  GroupKFold across 21 cities:
    pre_baseline_mean (2 feat)                    AUC=0.499 F1=0.027 P=0.122 R=0.015 [GroupKFold]
    saved oof -> oof_D1_blk_pre_baseline_mean__20260429_083322_06ea54.parquet  TP=125 FN=8122 FP=897 TN=54099
    post_block_mean (2 feat)                      AUC=0.499 F1=0.027 P=0.122 R=0.015 [GroupKFold]
    saved oof -> oof_D1_blk_post_block_mean__20260429_083322_06ea54.parquet  TP=125 FN=8122 FP=897 TN=54099
    pre_baseline_all (16 feat)                    AUC=0.524 F1=0.001 P=0.625 R=0.001 [GroupKFol

# DIETRICH REPLICATION: All-Blocks Approach
- Option A: Use ALL block columns as wide features (temporal contrast encoded in feature vector).Option B: True Dietrich dynamic labeling (long format, one row per building x block, label flips at damage date).

In [11]:
# @title CELL D1g: OPTION A — ALL BLOCKS WIDE, SINGLE CITY
# Use all blk01-blk14 + baseline as features. RF learns which block changed.
# Single-city StratifiedKFold (no geographic leakage).
import gc

print("=" * 70)
print("CELL D1g: ALL BLOCKS WIDE, SINGLE CITY")
print("=" * 70)

df_blk, feat_blk, _ = get_analysis_df('block_stats')
feat_blk = exclude_leakage(feat_blk)

# all SAR CARD features (exclude COH, keep all blocks + baseline)
sar_cols = [c for c in feat_blk if c.startswith('s1__v')]
nan_pct = df_blk[sar_cols].isna().mean()
sar_cols = [c for c in sar_cols if nan_pct[c] < 1.0]

# detect which blocks exist
blocks = sorted(set(c.split('__')[2] for c in sar_cols if len(c.split('__')) >= 4))
print(f"  SAR features: {len(sar_cols)}, blocks: {blocks}")

print(f"\n  Per-city RF (StratifiedKFold, all blocks as wide features):")
city_results = {}
for city in sorted(df_blk['city'].unique()):
    cdf = df_blk[df_blk['city'] == city]
    n_dam = (cdf[TARGET_COL] == 1).sum()
    if n_dam < 10 or (len(cdf) - n_dam) < 10:
        continue
    # only use columns that have data for this city
    city_cols = [c for c in sar_cols if cdf[c].notna().any()]
    if len(city_cols) < 5:
        continue
    X = cdf[city_cols].values
    y = cdf[TARGET_COL].values
    res = run_rf_cv(X, y, label=f"{city} ({n_dam} dam, {len(city_cols)} feat)")
    city_results[city] = res['auc']

if city_results:
    mean_auc = np.mean(list(city_results.values()))
    print(f"\n  Mean single-city AUC: {mean_auc:.3f} (across {len(city_results)} cities)")
    print(f"  Dietrich paper: {DIETRICH_PAPER['auc']:.3f}")

del df_blk; gc.collect()
print("  Memory freed")


CELL D1g: ALL BLOCKS WIDE, SINGLE CITY
  Loaded block_stats: 63243 rows, 330 features, 96.6 MB
  SAR features: 312, blocks: ['baseline', 'blk01', 'blk02', 'blk03', 'blk04', 'blk05', 'blk06', 'blk07', 'blk08', 'blk09', 'blk10', 'blk11', 'blk12', 'blk13', 'blk14', 'blk15', 'blk16', 'blk_pre01']

  Per-city RF (StratifiedKFold, all blocks as wide features):
    Avdiivka (93 dam, 288 feat)                   AUC=0.698 F1=0.000 P=0.000 R=0.000 [StratifiedKFold]
    Borodyanka (68 dam, 48 feat)                  AUC=0.633 F1=0.000 P=0.000 R=0.000 [StratifiedKFold]
    Bucha (194 dam, 48 feat)                      AUC=0.572 F1=0.000 P=0.000 R=0.000 [StratifiedKFold]
    Chernihiv (358 dam, 48 feat)                  AUC=0.540 F1=0.011 P=1.000 R=0.006 [StratifiedKFold]
    Chornobaivka (13 dam, 288 feat)               AUC=0.599 F1=0.000 P=0.000 R=0.000 [StratifiedKFold]
    Dmytrivka (361 dam, 48 feat)                  AUC=0.696 F1=0.359 P=0.341 R=0.380 [StratifiedKFold]
    Hostomel (618 dam, 48

In [12]:
# @title CELL D1h: OPTION B — DYNAMIC LABELING, SINGLE CITY
# True Dietrich: each building x block = one observation.
# Label = 0 if block ends before damage date, 1 if after.
# Uses scene_card (A2) per-scene long format.
import gc

print("=" * 70)
print("CELL D1h: DYNAMIC LABELING (SCENE_CARD), SINGLE CITY")
print("  Each building x date = one observation. Label from timestep.")
print("=" * 70)

df_card, feat_card, _ = get_analysis_df('scene_card')
feat_card = exclude_leakage(feat_card)

if 'timestep' not in df_card.columns:
    print("  SKIP: no timestep column in scene_card")
else:
    # dynamic label: t < 0 -> label=0 (pre-battle), t >= 0 -> label=1 (post-battle)
    # BUT we also need ground truth damage_binary to filter buildings
    # Only include buildings that are actually damaged (label=1) or undamaged (label=0)
    # For damaged buildings: pre-battle obs get label=0, post-battle get label=1
    # For undamaged buildings: ALL obs get label=0

    df_card['dynamic_label'] = 0
    is_damaged_building = df_card[TARGET_COL] == 1
    is_post_battle = df_card['timestep'] >= 0
    df_card.loc[is_damaged_building & is_post_battle, 'dynamic_label'] = 1

    print(f"  Dynamic labels: 0={( df_card['dynamic_label']==0).sum()}, 1={(df_card['dynamic_label']==1).sum()}")

    print(f"\n  Per-city RF (StratifiedKFold, dynamic labeling):")
    city_results = {}
    for city in sorted(df_card['city'].unique()):
        cdf = df_card[df_card['city'] == city]
        n_pos = (cdf['dynamic_label'] == 1).sum()
        n_neg = (cdf['dynamic_label'] == 0).sum()
        if n_pos < 20 or n_neg < 20:
            continue
        X = cdf[feat_card].values
        y = cdf['dynamic_label'].values
        res = run_rf_cv(X, y, label=f"{city} (pos={n_pos}, neg={n_neg})")
        city_results[city] = res['auc']

    if city_results:
        mean_auc = np.mean(list(city_results.values()))
        print(f"\n  Mean single-city AUC: {mean_auc:.3f} (across {len(city_results)} cities)")
        print(f"  Dietrich paper: {DIETRICH_PAPER['auc']:.3f}")
        print(f"\n  NOTE: This is change detection (pre vs post for damaged buildings),")
        print(f"  not damage classification. Dietrich uses this to TRAIN, then applies")
        print(f"  the trained model to unseen buildings to predict damage.")

del df_card; gc.collect()
print("  Memory freed")


CELL D1h: DYNAMIC LABELING (SCENE_CARD), SINGLE CITY
  Each building x date = one observation. Label from timestep.
  Loaded scene_card: 834134 rows, 2 features, 208.6 MB
  Dynamic labels: 0=768403, 1=65731

  Per-city RF (StratifiedKFold, dynamic labeling):
    Avdiivka (pos=5580, neg=71510)                AUC=0.581 F1=0.033 P=0.200 R=0.018 [StratifiedKFold]
    Borodyanka (pos=340, neg=4730)                AUC=0.577 F1=0.016 P=0.079 R=0.009 [StratifiedKFold]
    Bucha (pos=776, neg=14182)                    AUC=0.512 F1=0.007 P=0.103 R=0.004 [StratifiedKFold]
    Chernihiv (pos=1790, neg=26310)               AUC=0.529 F1=0.017 P=0.163 R=0.009 [StratifiedKFold]
    Chornobaivka (pos=299, neg=3535)              AUC=0.644 F1=0.115 P=0.313 R=0.070 [StratifiedKFold]
    Dmytrivka (pos=1444, neg=26780)               AUC=0.642 F1=0.217 P=0.151 R=0.381 [StratifiedKFold]
    Hostomel (pos=3090, neg=41880)                AUC=0.516 F1=0.018 P=0.158 R=0.010 [StratifiedKFold]
    Irpin (pos=1540,

In [13]:
# @title CELL D1i: OPTION A — ALL BLOCKS WIDE, MULTI CITY
# Same as D1g but GroupKFold across cities.
import gc

print("=" * 70)
print("CELL D1i: ALL BLOCKS WIDE, MULTI CITY (GroupKFold)")
print("=" * 70)

df_blk, feat_blk, _ = get_analysis_df('block_stats')
feat_blk = exclude_leakage(feat_blk)

sar_cols = [c for c in feat_blk if c.startswith('s1__v')]
nan_pct = df_blk[sar_cols].isna().mean()
sar_cols = [c for c in sar_cols if nan_pct[c] < 1.0]

groups = df_blk['city'].values
y = df_blk[TARGET_COL].values
n_cities = df_blk['city'].nunique()

print(f"  {len(sar_cols)} SAR features, {n_cities} cities, {len(df_blk)} buildings")

# Random CV vs GroupKFold
res_random = run_rf_cv(df_blk[sar_cols].values, y,
                       label=f"Random CV ({len(sar_cols)} feat)")
res_gkf = run_rf_cv(df_blk[sar_cols].values, y, groups=groups,
                     label=f"GroupKFold ({len(sar_cols)} feat)")

delta = res_random['auc'] - res_gkf['auc']
print(f"\n  Random CV:   AUC={res_random['auc']:.3f}")
print(f"  GroupKFold:  AUC={res_gkf['auc']:.3f}")
print(f"  Leakage delta: {delta:+.3f}")
print(f"  Dietrich paper: AUC={DIETRICH_PAPER['auc']:.3f}")

log_experiment('D1i_allblocks_random', 'block_stats', 'StratifiedKFold',
               res_random['auc'], res_random['f1'], len(sar_cols), n_cities, len(df_blk))
save_oof(res_random, 'D1i_allblocks_random',
         df_blk['point_id'].values, df_blk['city'].values, y,
         variant_id='random_cv')

log_experiment('D1i_allblocks_groupkfold', 'block_stats', 'GroupKFold',
               res_gkf['auc'], res_gkf['f1'], len(sar_cols), n_cities, len(df_blk),
               extra={'leakage_delta': delta})
save_oof(res_gkf, 'D1i_allblocks_groupkfold',
         df_blk['point_id'].values, df_blk['city'].values, y,
         variant_id='groupkfold_cv')

del df_blk; gc.collect()
print("  Memory freed")


CELL D1i: ALL BLOCKS WIDE, MULTI CITY (GroupKFold)
  Loaded block_stats: 63243 rows, 330 features, 96.6 MB
  312 SAR features, 21 cities, 63243 buildings
    Random CV (312 feat)                          AUC=0.632 F1=0.045 P=0.360 R=0.024 [StratifiedKFold]
    GroupKFold (312 feat)                         AUC=0.522 F1=0.001 P=1.000 R=0.001 [GroupKFold]

  Random CV:   AUC=0.632
  GroupKFold:  AUC=0.522
  Leakage delta: +0.110
  Dietrich paper: AUC=0.813
    saved oof -> oof_D1i_allblocks_random__20260429_083322_06ea54.parquet  TP=200 FN=8047 FP=355 TN=54641
    saved oof -> oof_D1i_allblocks_groupkfold__20260429_083322_06ea54.parquet  TP=6 FN=8241 FP=0 TN=54996
  Memory freed


In [14]:
# @title CELL D1j: OPTION B — DYNAMIC LABELING FROM BLOCKS, MULTI CITY (GroupKFold)
# Unpivot block_stats: one row per building x block, 14 features per row (7 stats x 2 pols).
# Dynamic label: block is pre-battle -> 0, block is post-battle AND building is damaged -> 1.
import gc

print("=" * 70)
print("CELL D1j: DYNAMIC LABELING (BLOCK_STATS UNPIVOTED), MULTI CITY")
print("=" * 70)

df_blk, feat_blk, _ = get_analysis_df('block_stats')
feat_blk = exclude_leakage(feat_blk)

# find all block names (blk01, blk02, ..., blk_pre01, baseline)
sar_cols = [c for c in feat_blk if c.startswith('s1__v')]
blocks = sorted(set(c.split('__')[2] for c in sar_cols if len(c.split('__')) >= 4))
print(f"  Blocks found: {blocks}")

# for each block, extract the stat suffix (e.g. mean_mean, std_max)
# column pattern: s1__vv__blk01__mean_mean -> pol=vv, block=blk01, stat=mean_mean
stat_suffixes = sorted(set(c.split('__')[3] for c in sar_cols if len(c.split('__')) >= 4))
# exclude count stats
stat_suffixes = [s for s in stat_suffixes if not s.startswith('count')]
print(f"  Stat suffixes: {len(stat_suffixes)} (excl count)")

# unpivot: for each block, create generic column names (vv__mean_mean, vh__std_max, ...)
# so RF sees the same feature names regardless of which block
generic_cols = []
for pol in ['vv', 'vh']:
    for stat in stat_suffixes:
        generic_cols.append(f's1__{pol}__{stat}')
print(f"  Generic columns per block: {len(generic_cols)}")

# build long dataframe
long_rows = []
meta_cols = ['point_id', 'city', TARGET_COL]

for block in blocks:
    # map block columns to generic names
    col_map = {}
    for pol in ['vv', 'vh']:
        for stat in stat_suffixes:
            src_col = f's1__{pol}__{block}__{stat}'
            dst_col = f's1__{pol}__{stat}'
            if src_col in df_blk.columns:
                col_map[src_col] = dst_col

    if len(col_map) < 4:
        continue

    block_df = df_blk[meta_cols + list(col_map.keys())].copy()
    block_df = block_df.rename(columns=col_map)
    block_df['block'] = block

    # determine block order: baseline=-1, blk_pre01=0, blk01=1, blk02=2, ...
    if block == 'baseline':
        block_df['block_idx'] = -1
    elif block == 'blk_pre01':
        block_df['block_idx'] = 0
    else:
        try:
            block_df['block_idx'] = int(block.replace('blk', ''))
        except ValueError:
            block_df['block_idx'] = 99

    long_rows.append(block_df)

df_long = pd.concat(long_rows, ignore_index=True)
del long_rows; gc.collect()

print(f"  Long format: {len(df_long)} rows ({df_long['point_id'].nunique()} buildings x {df_long['block'].nunique()} blocks)")

# dynamic labeling:
# baseline (block_idx=-1): always label=0
# blk_pre01 (block_idx=0): label=1 if damaged building (first cross-battle block)
# blk01+ (block_idx>=1): label=1 if damaged building (post-battle blocks)
df_long['dynamic_label'] = 0
is_damaged = df_long[TARGET_COL] == 1
is_post = df_long['block_idx'] >= 0
df_long.loc[is_damaged & is_post, 'dynamic_label'] = 1

n_pos = (df_long['dynamic_label'] == 1).sum()
n_neg = (df_long['dynamic_label'] == 0).sum()
n_cities = df_long['city'].nunique()
print(f"  Dynamic labels: pos={n_pos:,}, neg={n_neg:,}")

# drop rows where all generic features are NaN (block doesn't exist for this city)
feat_cols = [c for c in generic_cols if c in df_long.columns]
df_long = df_long.dropna(subset=feat_cols, how='all').reset_index(drop=True)
print(f"  After dropping all-NaN blocks: {len(df_long)} rows")

groups = df_long['city'].values
y = df_long['dynamic_label'].values

print(f"\n  {n_cities} cities, {len(df_long)} observations, {len(feat_cols)} features")

res_random = run_rf_cv(df_long[feat_cols].values, y,
                       label=f"Random CV ({len(feat_cols)} feat)")
res_gkf = run_rf_cv(df_long[feat_cols].values, y, groups=groups,
                     label=f"GroupKFold ({len(feat_cols)} feat)")

delta = res_random['auc'] - res_gkf['auc']
print(f"\n  Random CV:   AUC={res_random['auc']:.3f}")
print(f"  GroupKFold:  AUC={res_gkf['auc']:.3f}")
print(f"  Leakage delta: {delta:+.3f}")
print(f"  Dietrich paper: AUC={DIETRICH_PAPER['auc']:.3f}")

# per-city breakdown
print(f"\n  Per-city dynamic-label breakdown:")
for city in sorted(df_long['city'].unique()):
    cdf = df_long[df_long['city'] == city]
    n_p = (cdf['dynamic_label'] == 1).sum()
    n_n = (cdf['dynamic_label'] == 0).sum()
    n_blocks = cdf['block'].nunique()
    n_bldg_dam = cdf[cdf[TARGET_COL] == 1]['point_id'].nunique()
    print(f"    {city:25s} obs={len(cdf):>8d} pos={n_p:>6d} neg={n_n:>8d} blocks={n_blocks:>3d} dam_bldg={n_bldg_dam:>5d}")

log_experiment('D1j_dynamic_random', 'block_stats_unpivoted', 'StratifiedKFold',
               res_random['auc'], res_random['f1'], len(feat_cols), n_cities, len(df_long))
save_oof(res_random, 'D1j_dynamic_random',
         df_long['point_id'].values, df_long['city'].values, y,
         variant_id='random_cv')

log_experiment('D1j_dynamic_groupkfold', 'block_stats_unpivoted', 'GroupKFold',
               res_gkf['auc'], res_gkf['f1'], len(feat_cols), n_cities, len(df_long),
               extra={'leakage_delta': delta})
save_oof(res_gkf, 'D1j_dynamic_groupkfold',
         df_long['point_id'].values, df_long['city'].values, y,
         variant_id='groupkfold_cv')

del df_long, df_blk; gc.collect()
print("  Memory freed")

CELL D1j: DYNAMIC LABELING (BLOCK_STATS UNPIVOTED), MULTI CITY
  Loaded block_stats: 63243 rows, 330 features, 96.6 MB
  Blocks found: ['baseline', 'blk01', 'blk02', 'blk03', 'blk04', 'blk05', 'blk06', 'blk07', 'blk08', 'blk09', 'blk10', 'blk11', 'blk12', 'blk13', 'blk14', 'blk15', 'blk16', 'blk_pre01']
  Stat suffixes: 12 (excl count)
  Generic columns per block: 24
  Long format: 1138374 rows (63243 buildings x 18 blocks)
  Dynamic labels: pos=140,199, neg=998,175
  After dropping all-NaN blocks: 293116 rows

  21 cities, 293116 observations, 24 features
    Random CV (24 feat)                           AUC=0.602 F1=0.174 P=0.124 R=0.293 [StratifiedKFold]
    GroupKFold (24 feat)                          AUC=0.585 F1=0.172 P=0.125 R=0.275 [GroupKFold]

  Random CV:   AUC=0.602
  GroupKFold:  AUC=0.585
  Leakage delta: +0.017
  Dietrich paper: AUC=0.813

  Per-city dynamic-label breakdown:
    Avdiivka                  obs=   21348 pos=  1581 neg=   19767 blocks= 18 dam_bldg=   93
   

# CELL D1c: PRE-BASELINE vs POST SINGLE-SCENEDoes damage signal emerge only after battle? Compare pre-baseline composite AUC vs post-battle per-scene AUC.

In [15]:
# @title CELL D1c: PRE-BASELINE vs POST SINGLE-SCENE AUC
# Q: Is damage detectable from pre-battle imagery alone, or only after?
# Pre: composite_prepost_bands (A9) using ONLY prebattle features
# Post: scene_ms (A1) using ONLY postbattle observations
import gc

print("=" * 70)
print("CELL D1c: PRE-BASELINE vs POST SINGLE-SCENE")
print("=" * 70)

# --- Pre-battle features from composite (A9) ---
print("\n  --- PRE-BATTLE COMPOSITES (A9, prebattle features only) ---")
df_a9, feat_a9, _ = get_analysis_df('composite_prepost_bands')
feat_a9 = exclude_leakage(feat_a9)

pre_cols = [c for c in feat_a9 if 'prebattle' in c or 'winter_baseline' in c]
post_cols = [c for c in feat_a9 if 'post_winter' in c]
delta_cols = [c for c in feat_a9 if 'delta' in c.lower()]

groups = df_a9['city'].values
y = df_a9[TARGET_COL].values

print(f"  Pre-battle features: {len(pre_cols)}")
print(f"  Post-battle features: {len(post_cols)}")

results = {}
if len(pre_cols) >= 2:
    res_pre = run_rf_cv(df_a9[pre_cols].values, y, groups=groups,
                        label=f"Pre-battle only ({len(pre_cols)} feat)")
    results['pre_composite'] = res_pre['auc']
    log_experiment('D1c_pre_composite', 'composite_prepost_bands', 'GroupKFold',
                   res_pre['auc'], res_pre['f1'], len(pre_cols),
                   df_a9['city'].nunique(), len(df_a9))
    save_oof(res_pre, 'D1c_pre_composite',
             df_a9['point_id'].values, df_a9['city'].values, y,
             variant_id='pre_only')

if len(post_cols) >= 2:
    res_post = run_rf_cv(df_a9[post_cols].values, y, groups=groups,
                         label=f"Post-battle only ({len(post_cols)} feat)")
    results['post_composite'] = res_post['auc']
    log_experiment('D1c_post_composite', 'composite_prepost_bands', 'GroupKFold',
                   res_post['auc'], res_post['f1'], len(post_cols),
                   df_a9['city'].nunique(), len(df_a9))
    save_oof(res_post, 'D1c_post_composite',
             df_a9['point_id'].values, df_a9['city'].values, y,
             variant_id='post_only')

if len(pre_cols) >= 2 and len(post_cols) >= 2:
    res_both = run_rf_cv(df_a9[pre_cols + post_cols].values, y, groups=groups,
                         label=f"Pre + Post combined ({len(pre_cols + post_cols)} feat)")
    results['pre_plus_post'] = res_both['auc']

del df_a9; gc.collect()

# --- Post-battle per-scene (A1, postbattle obs only) ---
print("\n  --- POST-BATTLE SINGLE SCENES (A1, postbattle only) ---")
df_a1, feat_a1, _ = get_analysis_df('scene_ms')
feat_a1 = exclude_leakage(feat_a1)

if 'period_label' in df_a1.columns:
    df_post = df_a1[df_a1['period_label'] == 'postbattle'].copy()
    if TARGET_COL in df_post.columns and df_post[TARGET_COL].nunique() >= 2 and len(df_post) > 100:
        groups_post = df_post['city'].values
        y_post = df_post[TARGET_COL].values
        res_scene = run_rf_cv(df_post[feat_a1].values, y_post, groups=groups_post,
                              label=f"Post-battle scenes ({len(feat_a1)} feat, {len(df_post)} obs)")
        results['post_single_scene'] = res_scene['auc']
    else:
        print("    Insufficient postbattle data")
else:
    print("    No period_label column")

del df_a1; gc.collect()

print(f"\n  SUMMARY:")
for k, v in sorted(results.items()):
    print(f"    {k:30s} AUC={v:.3f}")
if 'pre_composite' in results and 'post_composite' in results:
    delta = results['post_composite'] - results['pre_composite']
    print(f"\n  Post - Pre delta: {delta:+.3f}")
    print(f"  {'Damage signal emerges post-battle' if delta > 0.03 else 'Pre-battle already discriminative (structural signal)'}")
print("  Memory freed")


CELL D1c: PRE-BASELINE vs POST SINGLE-SCENE

  --- PRE-BATTLE COMPOSITES (A9, prebattle features only) ---
  Loaded composite_prepost_bands: 62043 rows, 79 features, 32.5 MB
  Pre-battle features: 48
  Post-battle features: 15
    Pre-battle only (48 feat)                     AUC=0.643 F1=0.027 P=0.459 R=0.014 [GroupKFold]
    saved oof -> oof_D1c_pre_composite__20260429_083322_06ea54.parquet  TP=113 FN=7967 FP=133 TN=53830
    Post-battle only (15 feat)                    AUC=0.623 F1=0.046 P=0.496 R=0.024 [GroupKFold]
    saved oof -> oof_D1c_post_composite__20260429_083322_06ea54.parquet  TP=193 FN=7887 FP=196 TN=53767
    Pre + Post combined (63 feat)                 AUC=0.646 F1=0.030 P=0.432 R=0.016 [GroupKFold]

  --- POST-BATTLE SINGLE SCENES (A1, postbattle only) ---
  Loaded scene_ms: 695385 rows, 12 features, 202.0 MB
    Post-battle scenes (12 feat, 257226 obs)      AUC=0.661 F1=0.028 P=0.427 R=0.014 [GroupKFold]

  SUMMARY:
    post_composite                 AUC=0.623
    

# CELL D1d: PRE vs POST BASELINES (SAR)Same as D1c but for SAR CARD features. Tests if SAR pre-battle baseline already distinguishes buildings that will be damaged.

In [16]:
# @title CELL D1d: PRE vs POST SAR BASELINES
# Q: Do SAR pre-battle statistics already predict damage? (structural signature)
import gc

print("=" * 70)
print("CELL D1d: PRE vs POST SAR BASELINES")
print("  Parquet: prepost_single_card (A13) -- pre/post/delta VV+VH")
print("=" * 70)

df_a13, feat_a13, _ = get_analysis_df('prepost_single_card')
feat_a13 = exclude_leakage(feat_a13)

pre_cols = [c for c in feat_a13 if c.startswith('pre_')]
post_cols = [c for c in feat_a13 if c.startswith('post_')]
delta_cols = [c for c in feat_a13 if c.startswith('delta_')]

groups = df_a13['city'].values
y = df_a13[TARGET_COL].values

print(f"  Pre: {pre_cols}")
print(f"  Post: {post_cols}")
print(f"  Delta: {delta_cols}")

results = {}
for label, cols in [('pre_only', pre_cols), ('post_only', post_cols),
                    ('delta_only', delta_cols), ('pre+post+delta', pre_cols+post_cols+delta_cols)]:
    if len(cols) < 2:
        print(f"    {label}: too few features ({len(cols)})")
        continue
    res = run_rf_cv(df_a13[cols].values, y, groups=groups,
                    label=f"{label} ({len(cols)} feat)")
    results[label] = res['auc']
    log_experiment(f'D1d_{label}', 'prepost_single_card', 'GroupKFold',
                   res['auc'], res['f1'], len(cols), df_a13['city'].nunique(), len(df_a13))
    save_oof(res, f'D1d_{label}',
             df_a13['point_id'].values, df_a13['city'].values, y,
             variant_id=f'feat_set={label}')

print(f"\n  If pre_only AUC >> 0.5: pre-battle SAR encodes building structure (size, material)")
print(f"  If delta >> pre: change detection adds real damage signal")
print(f"  If post ~= pre+post+delta: delta features are redundant with pre+post")

del df_a13; gc.collect()
print("  Memory freed")


CELL D1d: PRE vs POST SAR BASELINES
  Parquet: prepost_single_card (A13) -- pre/post/delta VV+VH
  Loaded prepost_single_card: 63243 rows, 6 features, 16.7 MB
  Pre: ['pre_vv', 'pre_vh']
  Post: ['post_vv', 'post_vh']
  Delta: ['delta_vv', 'delta_vh']
    pre_only (2 feat)                             AUC=0.505 F1=0.030 P=0.138 R=0.017 [GroupKFold]
    saved oof -> oof_D1d_pre_only__20260429_083322_06ea54.parquet  TP=138 FN=8109 FP=865 TN=54131
    post_only (2 feat)                            AUC=0.506 F1=0.027 P=0.126 R=0.015 [GroupKFold]
    saved oof -> oof_D1d_post_only__20260429_083322_06ea54.parquet  TP=124 FN=8123 FP=862 TN=54134
    delta_only (2 feat)                           AUC=0.512 F1=0.028 P=0.137 R=0.016 [GroupKFold]
    saved oof -> oof_D1d_delta_only__20260429_083322_06ea54.parquet  TP=129 FN=8118 FP=810 TN=54186
    pre+post+delta (6 feat)                       AUC=0.514 F1=0.008 P=0.260 R=0.004 [GroupKFold]
    saved oof -> oof_D1d_pre+post+delta__20260429_083322_06

# CELL D1e: ZONAL AGGREGATION (per-pixel proxy)Compare _mean, _std, _max, _min zonal stats. Which aggregation captures damage best? _std should win (damaged buildings have heterogeneous pixel values).

In [17]:
# @title CELL D1e: ZONAL AGGREGATION COMPARISON
# Q: Which building-level summary of pixel values is most informative?
# _mean = average spectral value (building material)
# _std = within-building heterogeneity (rubble = mixed pixels)
# _max/_min = extreme values
import gc

print("=" * 70)
print("CELL D1e: ZONAL AGGREGATION COMPARISON")
print("  Parquet: block_stats (A15)")
print("=" * 70)

df_blk, feat_blk, _ = get_analysis_df('block_stats')
feat_blk = exclude_leakage(feat_blk)

# group by zonal aggregation suffix
agg_groups = {}
for c in feat_blk:
    parts = c.split('_')
    suffix = parts[-1] if parts else 'unknown'
    if suffix in ('mean', 'std', 'max', 'min', 'median'):
        agg_groups.setdefault(suffix, []).append(c)

# drop all-NaN cols per group
nan_pct = df_blk[feat_blk].isna().mean()
for k in agg_groups:
    agg_groups[k] = [c for c in agg_groups[k] if nan_pct.get(c, 1.0) < 1.0]

groups = df_blk['city'].values
y = df_blk[TARGET_COL].values

print(f"  Zonal aggregation groups:")
results = {}
for suffix in ['mean', 'std', 'max', 'min', 'median']:
    cols = agg_groups.get(suffix, [])
    if len(cols) < 2:
        print(f"    _{suffix}: {len(cols)} features (skip)")
        continue
    res = run_rf_cv(df_blk[cols].values, y, groups=groups,
                    label=f"_{suffix} only ({len(cols)} feat)")
    results[suffix] = res['auc']
    log_experiment(f'D1e_zonal_{suffix}', 'block_stats', 'GroupKFold',
                   res['auc'], res['f1'], len(cols), df_blk['city'].nunique(), len(df_blk))
    save_oof(res, f'D1e_zonal_{suffix}',
             df_blk['point_id'].values, df_blk['city'].values, y,
             variant_id=f'zonal={suffix}')

if results:
    best = max(results, key=results.get)
    print(f"\n  Best zonal aggregation: _{best} (AUC={results[best]:.3f})")
    print(f"  If _std wins: within-building heterogeneity is the damage signal (rubble, mixed materials)")
    print(f"  If _mean wins: absolute backscatter/reflectance level discriminates (building type confound?)")

del df_blk; gc.collect()
print("  Memory freed")


CELL D1e: ZONAL AGGREGATION COMPARISON
  Parquet: block_stats (A15)
  Loaded block_stats: 63243 rows, 330 features, 96.6 MB
  Zonal aggregation groups:
    _mean only (38 feat)                          AUC=0.512 F1=0.003 P=0.448 R=0.002 [GroupKFold]
    saved oof -> oof_D1e_zonal_mean__20260429_083322_06ea54.parquet  TP=13 FN=8234 FP=16 TN=54980
    _std only (38 feat)                           AUC=0.503 F1=0.001 P=0.167 R=0.000 [GroupKFold]
    saved oof -> oof_D1e_zonal_std__20260429_083322_06ea54.parquet  TP=3 FN=8244 FP=15 TN=54981
    _max only (44 feat)                           AUC=0.524 F1=0.006 P=0.610 R=0.003 [GroupKFold]
    saved oof -> oof_D1e_zonal_max__20260429_083322_06ea54.parquet  TP=25 FN=8222 FP=16 TN=54980
    _min only (44 feat)                           AUC=0.520 F1=0.007 P=0.574 R=0.003 [GroupKFold]
    saved oof -> oof_D1e_zonal_min__20260429_083322_06ea54.parquet  TP=27 FN=8220 FP=20 TN=54976
    _median only (38 feat)                        AUC=0.517 F1=0.002

# CELL D1f: ROLLING WINDOW COMPARISONCompare roll3 vs roll7 vs roll13 assessment features. Shorter windows = sharper signal but fewer observations. Longer = smoother but may blur damage onset.

In [18]:
# @title CELL D1f: ROLLING WINDOW COMPARISON (A16 vs A17 vs A18)
# Q: Does rolling window size matter for damage detection?
import gc

print("=" * 70)
print("CELL D1f: ROLLING WINDOW COMPARISON")
print("=" * 70)

results = {}

for pq_name, label, roll_n in [('rolling_stats_roll3', 'roll3', 3),
                                 ('rolling_stats_roll7', 'roll7', 7),
                                 ('rolling_stats_roll13', 'roll13', 13)]:
    print(f"\n  --- {label} ({pq_name}) ---")
    df_pq, feat_pq, _ = get_analysis_df(pq_name)
    feat_pq = exclude_leakage(feat_pq)
    nan_pct = df_pq[feat_pq].isna().mean()
    feat_pq = [c for c in feat_pq if nan_pct[c] < 1.0]

    # separate baseline (shared) vs assessment (window-specific)
    baseline_cols = [c for c in feat_pq if '__baseline__' in c]
    assess_cols = [c for c in feat_pq if f'__roll{roll_n}__' in c]

    groups = df_pq['city'].values
    y = df_pq[TARGET_COL].values

    # assessment features only (these differ per window)
    if len(assess_cols) >= 2:
        res_assess = run_rf_cv(df_pq[assess_cols].values, y, groups=groups,
                               label=f"{label} assessment only ({len(assess_cols)} feat)")
        results[f'{label}_assess'] = res_assess['auc']
        log_experiment(f'D1f_{label}_assess', pq_name, 'GroupKFold',
                       res_assess['auc'], res_assess['f1'], len(assess_cols),
                       df_pq['city'].nunique(), len(df_pq))
        save_oof(res_assess, f'D1f_{label}_assess',
                 df_pq['point_id'].values, df_pq['city'].values, y,
                 variant_id=f'window={label}_assess_only')

    # all features (baseline + assessment)
    if len(feat_pq) >= 2:
        res_all = run_rf_cv(df_pq[feat_pq].values, y, groups=groups,
                            label=f"{label} all ({len(feat_pq)} feat)")
        results[f'{label}_all'] = res_all['auc']

    del df_pq; gc.collect()

# Fusion with MS: fusion_composite_cohdrop (F7) already has coh_drop + MS
# Compare with SAR-only rolling
print(f"\n  --- Fusion: MS + COH drop (F7) ---")
df_f7, feat_f7, _ = get_analysis_df('fusion_composite_cohdrop')
feat_f7 = exclude_leakage(feat_f7)
nan_pct = df_f7[feat_f7].isna().mean()
feat_f7 = [c for c in feat_f7 if nan_pct[c] < 1.0]
groups = df_f7['city'].values
y = df_f7[TARGET_COL].values
res_fusion = run_rf_cv(df_f7[feat_f7].values, y, groups=groups,
                       label=f"F7 fusion ({len(feat_f7)} feat)")
results['fusion_f7'] = res_fusion['auc']
del df_f7; gc.collect()

print(f"\n  SUMMARY:")
for k, v in sorted(results.items()):
    print(f"    {k:30s} AUC={v:.3f}")
print(f"\n  If roll3_assess > roll13_assess: shorter window captures sharper damage onset")
print(f"  If roll13_assess > roll3_assess: longer window accumulates more evidence")
print(f"  If fusion_f7 >> any roll: MS features add value beyond SAR")
print("  Memory freed")


CELL D1f: ROLLING WINDOW COMPARISON

  --- roll3 (rolling_stats_roll3) ---
  Loaded rolling_stats_roll3: 63243 rows, 38 features, 22.8 MB
    roll3 assessment only (12 feat)               AUC=0.478 F1=0.171 P=0.120 R=0.296 [GroupKFold]
    saved oof -> oof_D1f_roll3_assess__20260429_083322_06ea54.parquet  TP=2440 FN=5807 FP=17884 TN=37112
    roll3 all (34 feat)                           AUC=0.515 F1=0.001 P=0.556 R=0.001 [GroupKFold]

  --- roll7 (rolling_stats_roll7) ---
  Loaded rolling_stats_roll7: 63243 rows, 38 features, 22.8 MB
    roll7 assessment only (2 feat)                AUC=0.478 F1=0.171 P=0.120 R=0.296 [GroupKFold]
    saved oof -> oof_D1f_roll7_assess__20260429_083322_06ea54.parquet  TP=2440 FN=5807 FP=17884 TN=37112
    roll7 all (24 feat)                           AUC=0.520 F1=0.001 P=0.429 R=0.000 [GroupKFold]

  --- roll13 (rolling_stats_roll13) ---
  Loaded rolling_stats_roll13: 63243 rows, 22 features, 18.7 MB
    roll13 all (22 feat)                          AUC

# CELL D1b: DIETRICH-28 BLOCK DIAGNOSTIC (bda_block_stats)TRUE Dietrich replication using 3-month block windows from NB03e P1c.- Pre: P1 baseline (s1__{pol}__baseline__{stat}) — 1-year pre-war- Post: blk01 (s1__{pol}__blk01__{stat}) — first 3 months from battle_start- Total: 2 pol × 7 stats × 2 periods × 1 zonal(_mean) = 28 featuresThis fixes the product_prepost assessment NaN problem: P1 assessment requires scenesAFTER battle_end + proximity buffer (only 2 scenes < min_obs=3 → all NaN).Block blk01 uses scenes DURING the battle period (7 scenes for Mariupol).

In [19]:
# @title CELL D2: MULTIMODAL EXTENSION (F8 = MS + SAR block stats)
# Goal: Does adding MS composites to SAR block stats improve GroupKFold AUC?
import gc

print("=" * 70)
print("CELL D2: MULTIMODAL EXTENSION")
print("  Parquet: fusion_composite_blockstats (F8)")
print("=" * 70)

df_f8, feat_f8, _ = get_analysis_df('fusion_composite_blockstats')
feat_clean = exclude_leakage(feat_f8)
print(f"  Features: {len(feat_f8)} total, {len(feat_clean)} after leakage exclusion")

# separate into modality groups
sar_cols = [c for c in feat_clean if c.startswith('s1__v')]
ms_cols = [c for c in feat_clean if c.startswith('s2__composite__')]
coh_cols = [c for c in feat_clean if c.startswith('s1__coh__')]

# drop all-NaN
nan_pct = df_f8[feat_clean].isna().mean()
feat_clean = [c for c in feat_clean if nan_pct[c] < 1.0]
sar_cols = [c for c in sar_cols if c in feat_clean]
ms_cols = [c for c in ms_cols if c in feat_clean]
coh_cols = [c for c in coh_cols if c in feat_clean]

print(f"  SAR CARD: {len(sar_cols)}, MS composite: {len(ms_cols)}, COH: {len(coh_cols)}")

# sensor ablation with GroupKFold
groups = df_f8['city'].values
y = df_f8[TARGET_COL].values

ablation_sets = {
    'SAR-only (Dietrich equivalent)': sar_cols,
    'MS-only (optical composites)': ms_cols,
    'COH-only (coherence baselines)': coh_cols,
    'SAR + MS (multimodal)': sar_cols + ms_cols,
    'SAR + MS + COH (full)': sar_cols + ms_cols + coh_cols,
    'All features (clean)': feat_clean,
}

print(f"\n  Sensor ablation (GroupKFold, {df_f8['city'].nunique()} cities):")
for label, cols in ablation_sets.items():
    if len(cols) < 2:
        print(f"    {label:45s} SKIP ({len(cols)} features)")
        continue
    X = df_f8[cols].values
    res = run_rf_cv(X, y, groups=groups, label=f"{label} ({len(cols)} feat)")
    short_id = label.split("(")[0].strip().replace(" ", "_").lower()
    log_experiment(f'D2_{short_id}',
                   'fusion_composite_blockstats', 'GroupKFold',
                   res['auc'], res['f1'], len(cols),
                   df_f8['city'].nunique(), len(df_f8))
    save_oof(res, f'D2_{short_id}',
             df_f8['point_id'].values, df_f8['city'].values, y,
             variant_id=f'ablation={label}')

print(f"\n  Dietrich paper reference: AUC={DIETRICH_PAPER['auc']:.3f} (SAR-only, random CV)")

del df_f8; gc.collect()
print("  Memory freed")


CELL D2: MULTIMODAL EXTENSION
  Parquet: fusion_composite_blockstats (F8)
  Loaded fusion_composite_blockstats: 62043 rows, 409 features, 115.4 MB
  Features: 409 total, 399 after leakage exclusion
  SAR CARD: 312, MS composite: 60, COH: 12

  Sensor ablation (GroupKFold, 19 cities):
    SAR-only (Dietrich equivalent) (312 feat)     AUC=0.533 F1=0.036 P=0.315 R=0.019 [GroupKFold]
    saved oof -> oof_D2_sar-only__20260429_083322_06ea54.parquet  TP=152 FN=7928 FP=331 TN=53632
    MS-only (optical composites) (60 feat)        AUC=0.646 F1=0.026 P=0.437 R=0.014 [GroupKFold]
    saved oof -> oof_D2_ms-only__20260429_083322_06ea54.parquet  TP=110 FN=7970 FP=142 TN=53821
    COH-only (coherence baselines) (12 feat)      AUC=0.491 F1=0.031 P=0.116 R=0.018 [GroupKFold]
    saved oof -> oof_D2_coh-only__20260429_083322_06ea54.parquet  TP=145 FN=7935 FP=1109 TN=52854
    SAR + MS (multimodal) (372 feat)              AUC=0.655 F1=0.008 P=0.393 R=0.004 [GroupKFold]
    saved oof -> oof_D2_sar_+_ms

In [20]:
# @title CELL D3: DELIBERATE LEAKAGE DEMONSTRATION
# Goal: Prove leakage mechanism by INCLUDING scenes_observed, then showing via SHAP
import gc

print("=" * 70)
print("CELL D3: DELIBERATE LEAKAGE DEMONSTRATION")
print("  Parquet: fusion_composite_cohdrop (F7) -- WITH leakage features")
print("=" * 70)

df_f7, feat_f7, _ = get_analysis_df('fusion_composite_cohdrop')
feat_clean = exclude_leakage(feat_f7)
feat_leaky = feat_f7  # keep ALL including scenes_observed

# drop all-NaN
nan_pct = df_f7[feat_leaky].isna().mean()
feat_leaky = [c for c in feat_leaky if nan_pct[c] < 1.0]
feat_clean = [c for c in feat_clean if nan_pct.get(c, 1.0) < 1.0]

groups = df_f7['city'].values
y = df_f7[TARGET_COL].values

print(f"  Clean features: {len(feat_clean)}")
print(f"  Leaky features: {len(feat_leaky)} (includes scenes_observed, obs_count, cloud_freq)")

print(f"\n  GroupKFold comparison:")
res_clean = run_rf_cv(df_f7[feat_clean].values, y, groups=groups, label="CLEAN (leakage excluded)")
res_leaky = run_rf_cv(df_f7[feat_leaky].values, y, groups=groups, label="LEAKY (all features)")

delta = res_leaky['auc'] - res_clean['auc']
print(f"\n  Leakage inflation: {delta:+.3f} AUC ({delta/res_clean['auc']*100:+.1f}%)")
print(f"  Clean AUC:  {res_clean['auc']:.3f}")
print(f"  Leaky AUC:  {res_leaky['auc']:.3f}")

# feature importance comparison
from sklearn.ensemble import RandomForestClassifier
imp_obj = SimpleImputer(strategy='median')
X_leaky = imp_obj.fit_transform(df_f7[feat_leaky].values)
rf_leaky = RandomForestClassifier(**RF_PARAMS)
rf_leaky.fit(X_leaky, y)
importances = pd.Series(rf_leaky.feature_importances_, index=feat_leaky).sort_values(ascending=False)

print(f"\n  Top 10 features by RF importance (LEAKY model):")
for feat, imp in importances.head(10).items():
    is_leakage = any(re.match(p, feat) for p in EXCLUDE_PATTERNS)
    tag = " <-- LEAKAGE" if is_leakage else ""
    print(f"    {feat:55s} imp={imp:.4f}{tag}")

n_leakage_in_top10 = sum(1 for f in importances.head(10).index if any(re.match(p, f) for p in EXCLUDE_PATTERNS))
print(f"\n  {n_leakage_in_top10}/10 top features are leakage vectors")

log_experiment('D3_leaky', 'fusion_composite_cohdrop', 'GroupKFold',
               res_leaky['auc'], res_leaky['f1'], len(feat_leaky),
               df_f7['city'].nunique(), len(df_f7), extra={'leakage_inflation': delta})
save_oof(res_leaky, 'D3_leaky',
         df_f7['point_id'].values, df_f7['city'].values, y,
         variant_id='leakage_demo_with_leaks')

log_experiment('D3_clean', 'fusion_composite_cohdrop', 'GroupKFold',
               res_clean['auc'], res_clean['f1'], len(feat_clean),
               df_f7['city'].nunique(), len(df_f7))
save_oof(res_clean, 'D3_clean',
         df_f7['point_id'].values, df_f7['city'].values, y,
         variant_id='leakage_demo_clean')

del df_f7, rf_leaky, X_leaky; gc.collect()
print("  Memory freed")


CELL D3: DELIBERATE LEAKAGE DEMONSTRATION
  Parquet: fusion_composite_cohdrop (F7) -- WITH leakage features
  Loaded fusion_composite_cohdrop: 51293 rows, 86 features, 29.2 MB
  Clean features: 75
  Leaky features: 86 (includes scenes_observed, obs_count, cloud_freq)

  GroupKFold comparison:
    CLEAN (leakage excluded)                      AUC=0.619 F1=0.020 P=0.439 R=0.010 [GroupKFold]
    LEAKY (all features)                          AUC=0.617 F1=0.019 P=0.523 R=0.010 [GroupKFold]

  Leakage inflation: -0.002 AUC (-0.2%)
  Clean AUC:  0.619
  Leaky AUC:  0.617

  Top 10 features by RF importance (LEAKY model):
    s2__composite__ndvi__post_winter_baseline               imp=0.0297
    s2__composite__ndwi__post_winter_baseline               imp=0.0278
    s2__composite__mndwi__prebattle_baseline                imp=0.0242
    s2__composite__ndvi__prebattle_baseline                 imp=0.0240
    s2__composite__b02__post_winter_baseline                imp=0.0231
    s2__composite__ndwi

# CELL D4: NaN DIAGNOSTICShows how NaN row-drop destroys SAR signal (NB09e finding).

In [21]:
# @title CELL D4: NaN STRATEGY COMPARISON (A9 vs F8)
# Goal: Show NaN drop vs impute impact on AUC (NB09e key finding preview)
import gc

print("=" * 70)
print("CELL D4: NaN STRATEGY COMPARISON")
print("=" * 70)

# A9: 0% NaN (MS-only composites)
df_a9, feat_a9, _ = get_analysis_df('composite_prepost_bands')
feat_a9 = exclude_leakage(feat_a9)
y_a9 = df_a9[TARGET_COL].values
groups_a9 = df_a9['city'].values

print(f"\n  A9 (composite_prepost_bands): {len(feat_a9)} features, 0% NaN")
res_a9 = run_rf_cv(df_a9[feat_a9].values, y_a9, groups=groups_a9,
                    label="A9 MS-only (no NaN, GroupKFold)")
save_oof(res_a9, 'D4_a9_ms_only',
         df_a9['point_id'].values, df_a9['city'].values, y_a9,
         variant_id='nan_strategy=no_nan_baseline')

del df_a9; gc.collect()

# F8: structural NaN (block_stats only for 6 cities)
df_f8, feat_f8, _ = get_analysis_df('fusion_composite_blockstats')
feat_f8 = exclude_leakage(feat_f8)
nan_pct = df_f8[feat_f8].isna().mean()
feat_f8 = [c for c in feat_f8 if nan_pct[c] < 1.0]
y_f8 = df_f8[TARGET_COL].values
groups_f8 = df_f8['city'].values

pct_nan_rows = df_f8[feat_f8].isna().any(axis=1).mean()
print(f"\n  F8 (fusion_composite_blockstats): {len(feat_f8)} features, {pct_nan_rows:.0%} rows with NaN")

# Strategy A: drop NaN rows
mask_a = ~df_f8[feat_f8].isna().any(axis=1).values
if mask_a.sum() > 50 and len(np.unique(y_f8[mask_a])) > 1:
    n_cities_a = len(np.unique(groups_f8[mask_a]))
    res_drop = run_rf_cv(df_f8.loc[mask_a, feat_f8].values, y_f8[mask_a],
                         groups=groups_f8[mask_a] if n_cities_a >= 2 else None,
                         label=f"F8 drop-NaN ({mask_a.sum()} rows, {n_cities_a} cities)")
    save_oof(res_drop, 'D4_f8_strategy_A_drop_nan',
             df_f8.loc[mask_a, 'point_id'].values,
             df_f8.loc[mask_a, 'city'].values,
             y_f8[mask_a],
             variant_id='nan_strategy=A_drop_rows')
else:
    print("    F8 drop-NaN: too few rows")

# Strategy B: median impute (keep all rows)
res_impute = run_rf_cv(df_f8[feat_f8].values, y_f8, groups=groups_f8,
                       label=f"F8 impute ({len(y_f8)} rows, {df_f8['city'].nunique()} cities)")
save_oof(res_impute, 'D4_f8_strategy_B_impute',
         df_f8['point_id'].values, df_f8['city'].values, y_f8,
         variant_id='nan_strategy=B_impute_all_rows')

print(f"\n  Key insight: NaN drop loses {(1-mask_a.mean())*100:.0f}% of data")
print(f"  If drop-NaN AUC >> impute AUC: NaN drop selects easy-to-classify cities (leakage)")
print(f"  If impute AUC >> drop-NaN AUC: imputation preserves cross-city signal")

del df_f8; gc.collect()
print("  Memory freed")


CELL D4: NaN STRATEGY COMPARISON
  Loaded composite_prepost_bands: 62043 rows, 79 features, 32.5 MB

  A9 (composite_prepost_bands): 69 features, 0% NaN
    A9 MS-only (no NaN, GroupKFold)               AUC=0.646 F1=0.025 P=0.448 R=0.013 [GroupKFold]
    saved oof -> oof_D4_a9_ms_only__20260429_083322_06ea54.parquet  TP=104 FN=7976 FP=128 TN=53835
  Loaded fusion_composite_blockstats: 62043 rows, 409 features, 115.4 MB

  F8 (fusion_composite_blockstats): 393 features, 100% rows with NaN
    F8 drop-NaN: too few rows
    F8 impute (62043 rows, 19 cities)             AUC=0.646 F1=0.009 P=0.365 R=0.004 [GroupKFold]
    saved oof -> oof_D4_f8_strategy_B_impute__20260429_083322_06ea54.parquet  TP=35 FN=8045 FP=61 TN=53902

  Key insight: NaN drop loses 100% of data
  If drop-NaN AUC >> impute AUC: NaN drop selects easy-to-classify cities (leakage)
  If impute AUC >> drop-NaN AUC: imputation preserves cross-city signal
  Memory freed


# CELL D5: NEIGHBORHOOD LEARNING DEMOSame-location controls vs random controls.Shows AUC=1.0 vs ~0.79 = Dietrich learns 'change happened' not 'damage pattern'.

In [22]:
# @title CELL D5: NEIGHBORHOOD LEARNING DEMO
# Goal: Show RF exploits spatial proximity, not just spectral features
import gc

print("=" * 70)
print("CELL D5: NEIGHBORHOOD LEARNING DEMO")
print("  Parquet: rolling_stats_roll7 (A17) -- SAR features, all 21 cities")
print("=" * 70)

df_a17, feat_a17, _ = get_analysis_df('rolling_stats_roll7')
feat_clean = exclude_leakage(feat_a17)

# V3: use x_utm/y_utm as primary coordinates; fall back to centroid_x/y (V2)
if 'x_utm' in df_a17.columns and 'y_utm' in df_a17.columns:
    _cx33, _cy33 = 'x_utm', 'y_utm'
elif 'centroid_x' in df_a17.columns and 'centroid_y' in df_a17.columns:
    _cx33, _cy33 = 'centroid_x', 'centroid_y'
else:
    _cx33, _cy33 = None, None
if _cx33 is None:
    print("  SKIP: no centroid columns in merged data")
else:
    damaged = df_a17[df_a17[TARGET_COL] == 1]
    undamaged = df_a17[df_a17[TARGET_COL] == 0]

    n_sample = min(len(damaged), len(undamaged))
    np.random.seed(RANDOM_STATE)
    random_ctrl = undamaged.sample(n=n_sample, random_state=RANDOM_STATE)
    df_random = pd.concat([damaged.head(n_sample), random_ctrl])

    X = df_random[feat_clean].values
    y = df_random[TARGET_COL].values
    res_random = run_rf_cv(X, y, label="Random controls (geographic mix)")

    # same-location controls: pair nearest damaged/undamaged
    from scipy.spatial import cKDTree
    dmg_xy = damaged[[_cx33, _cy33]].values
    undmg_xy = undamaged[[_cx33, _cy33]].values
    if len(dmg_xy) > 0 and len(undmg_xy) > 0:
        tree = cKDTree(undmg_xy)
        _, idx = tree.query(dmg_xy, k=1)
        near_ctrl = undamaged.iloc[idx]
        df_near = pd.concat([damaged, near_ctrl])
        X2 = df_near[feat_clean].values
        y2 = df_near[TARGET_COL].values
        res_near = run_rf_cv(X2, y2, label="Same-location controls (neighbors)")

        print(f"\n  If neighbor AUC < random AUC:")
        print(f"  -> model partly exploits geographic separation, not just damage features")
        print(f"  -> supports GroupKFold requirement")

del df_a17; gc.collect()
print("  Memory freed")


CELL D5: NEIGHBORHOOD LEARNING DEMO
  Parquet: rolling_stats_roll7 (A17) -- SAR features, all 21 cities
  Loaded rolling_stats_roll7: 63243 rows, 38 features, 22.8 MB
    Random controls (geographic mix)              AUC=0.556 F1=0.537 P=0.541 R=0.533 [StratifiedKFold]
    Same-location controls (neighbors)            AUC=0.810 F1=0.768 P=0.717 R=0.826 [StratifiedKFold]

  If neighbor AUC < random AUC:
  -> model partly exploits geographic separation, not just damage features
  -> supports GroupKFold requirement
  Memory freed


# CELL D6: MULTIMODAL EXTENSION (Mariupol)Add COH + MS to Dietrich-28. Single-city proof that multimodal helps.

In [23]:
# @title CELL D6: SENSOR ABLATION (MS vs SAR vs Fusion, GroupKFold)
# Goal: Which sensor modality contributes most to honest cross-city AUC?
import gc

print("=" * 70)
print("CELL D6: SENSOR ABLATION")
print("=" * 70)

# MS-only: composite_prepost_bands (A9)
print("\n  --- MS-only (A9) ---")
df_a9, feat_a9, _ = get_analysis_df('composite_prepost_bands')
feat_a9 = exclude_leakage(feat_a9)
groups_a9 = df_a9['city'].values
y_a9 = df_a9[TARGET_COL].values
res_ms = run_rf_cv(df_a9[feat_a9].values, y_a9, groups=groups_a9,
                   label=f"MS-only ({len(feat_a9)} feat, {df_a9['city'].nunique()} cities)")
log_experiment('D6_ms_only', 'composite_prepost_bands', 'GroupKFold',
               res_ms['auc'], res_ms['f1'], len(feat_a9), df_a9['city'].nunique(), len(df_a9))
save_oof(res_ms, 'D6_ms_only',
         df_a9['point_id'].values, df_a9['city'].values, y_a9,
         variant_id='sensor=MS_only')
del df_a9; gc.collect()

# SAR-only: rolling_stats_roll7 (A17)
print("\n  --- SAR-only (A17) ---")
df_a17, feat_a17, _ = get_analysis_df('rolling_stats_roll7')
feat_a17 = exclude_leakage(feat_a17)
groups_a17 = df_a17['city'].values
y_a17 = df_a17[TARGET_COL].values
res_sar = run_rf_cv(df_a17[feat_a17].values, y_a17, groups=groups_a17,
                    label=f"SAR-only ({len(feat_a17)} feat, {df_a17['city'].nunique()} cities)")
log_experiment('D6_sar_only', 'rolling_stats_roll7', 'GroupKFold',
               res_sar['auc'], res_sar['f1'], len(feat_a17), df_a17['city'].nunique(), len(df_a17))
save_oof(res_sar, 'D6_sar_only',
         df_a17['point_id'].values, df_a17['city'].values, y_a17,
         variant_id='sensor=SAR_only')
del df_a17; gc.collect()

# Fusion: fusion_composite_cohdrop (F7)
print("\n  --- Fusion MS+SAR (F7) ---")
df_f7, feat_f7, _ = get_analysis_df('fusion_composite_cohdrop')
feat_f7 = exclude_leakage(feat_f7)
nan_pct = df_f7[feat_f7].isna().mean()
feat_f7 = [c for c in feat_f7 if nan_pct[c] < 1.0]
groups_f7 = df_f7['city'].values
y_f7 = df_f7[TARGET_COL].values
res_fusion = run_rf_cv(df_f7[feat_f7].values, y_f7, groups=groups_f7,
                       label=f"Fusion ({len(feat_f7)} feat, {df_f7['city'].nunique()} cities)")
log_experiment('D6_fusion', 'fusion_composite_cohdrop', 'GroupKFold',
               res_fusion['auc'], res_fusion['f1'], len(feat_f7), df_f7['city'].nunique(), len(df_f7))
save_oof(res_fusion, 'D6_fusion',
         df_f7['point_id'].values, df_f7['city'].values, y_f7,
         variant_id='sensor=fusion_MS_SAR_COH')
del df_f7; gc.collect()

print(f"\n  Summary:")
print(f"    MS-only:  AUC={res_ms['auc']:.3f}")
print(f"    SAR-only: AUC={res_sar['auc']:.3f}")
print(f"    Fusion:   AUC={res_fusion['auc']:.3f}")
print(f"    Dietrich: AUC={DIETRICH_PAPER['auc']:.3f} (SAR-only, random CV, not comparable)")
print("  Memory freed")


CELL D6: SENSOR ABLATION

  --- MS-only (A9) ---
  Loaded composite_prepost_bands: 62043 rows, 79 features, 32.5 MB
    MS-only (69 feat, 19 cities)                  AUC=0.646 F1=0.025 P=0.448 R=0.013 [GroupKFold]
    saved oof -> oof_D6_ms_only__20260429_083322_06ea54.parquet  TP=104 FN=7976 FP=128 TN=53835

  --- SAR-only (A17) ---
  Loaded rolling_stats_roll7: 63243 rows, 38 features, 22.8 MB
    SAR-only (38 feat, 21 cities)                 AUC=0.520 F1=0.001 P=0.429 R=0.000 [GroupKFold]
    saved oof -> oof_D6_sar_only__20260429_083322_06ea54.parquet  TP=3 FN=8244 FP=4 TN=54992

  --- Fusion MS+SAR (F7) ---
  Loaded fusion_composite_cohdrop: 51293 rows, 86 features, 29.2 MB
    Fusion (75 feat, 11 cities)                   AUC=0.619 F1=0.020 P=0.439 R=0.010 [GroupKFold]
    saved oof -> oof_D6_fusion__20260429_083322_06ea54.parquet  TP=69 FN=6798 FP=88 TN=44338

  Summary:
    MS-only:  AUC=0.646
    SAR-only: AUC=0.520
    Fusion:   AUC=0.619
    Dietrich: AUC=0.813 (SAR-only, ra

# CELL D7: TEMPORAL TRAJECTORY VISUALIZATIONPlot per-building CARD/COH time series colored by damage. NO model.

In [24]:
# @title CELL D7: TEMPORAL TRAJECTORY VISUALIZATION (bda_scene_card + bda_scene_coh)
import matplotlib.pyplot as plt

print("=" * 70)
print("CELL D7: TEMPORAL TRAJECTORY VISUALIZATION")
print("=" * 70)

# load battle dates for vertical markers
battle_dates = {}
for city in CITIES_TO_PROCESS:
    aoi_path = CITIES_DIR / city / 'AOI.geojson'
    if aoi_path.exists():
        import re
        with open(aoi_path) as f:
            raw = f.read(50000)
        m = re.search(r'"battle_start"\s*:\s*"([^"]+)"', raw)
        if m:
            battle_dates[city] = pd.Timestamp(m.group(1))
if battle_dates:
    print(f"  Battle dates loaded for {len(battle_dates)} cities")

n_sample = 20


def plot_trajectories(df, value_col, date_col, sensor_label):
    """Plot damaged vs undamaged trajectories per city with battle_start marker."""
    cities = sorted(df['city'].unique())
    n_cities = len(cities)
    fig, axes = plt.subplots(n_cities, 2, figsize=(14, 4 * n_cities), squeeze=False)

    for row, city in enumerate(cities):
        df_city = df[df['city'] == city]
        dmg_ids = df_city[df_city[TARGET_COL] == 1]['point_id'].unique()
        undmg_ids = df_city[df_city[TARGET_COL] == 0]['point_id'].unique()
        np.random.seed(RANDOM_STATE)
        s_dmg = np.random.choice(dmg_ids, min(n_sample, len(dmg_ids)), replace=False) if len(dmg_ids) > 0 else []
        s_undmg = np.random.choice(undmg_ids, min(n_sample, len(undmg_ids)), replace=False) if len(undmg_ids) > 0 else []

        for col_idx, (ids, title, color) in enumerate([
            (s_undmg, 'Undamaged', 'blue'),
            (s_dmg, 'Damaged', 'red'),
        ]):
            ax = axes[row, col_idx]
            vals_by_date = {}
            for bid in ids:
                bdata = df_city[df_city['point_id'] == bid].sort_values(date_col)
                dates = bdata[date_col].values
                vals = bdata[value_col].values
                ax.plot(dates, vals, alpha=0.2, color=color, linewidth=0.5)
                for d, v in zip(dates, vals):
                    vals_by_date.setdefault(d, []).append(v)

            # median trajectory overlay
            if vals_by_date:
                sorted_dates = sorted(vals_by_date.keys())
                medians = [np.nanmedian(vals_by_date[d]) for d in sorted_dates]
                ax.plot(sorted_dates, medians, color=color, linewidth=2, alpha=0.8, label='median')

            # battle_start vertical line
            if city in battle_dates:
                ax.axvline(battle_dates[city], color='black', linestyle='--', linewidth=1, label='battle_start')

            ax.set_title(f'{city} - {title} ({sensor_label})')
            ax.set_xlabel('Date')
            ax.set_ylabel(value_col)
            ax.tick_params(axis='x', rotation=45)
            if col_idx == 0 and row == 0:
                ax.legend(fontsize=8)

    plt.tight_layout()
    save_fig(fig, f'trajectory_{sensor_label.lower().replace(" ", "_")}', 'cell_d7')


# CARD trajectories
try:
    df_sc = load_v3_parquet('scene_card')
    city_list = [c for c in CITIES_TO_PROCESS if c in df_sc['city'].unique()]
    df_sc = df_sc[df_sc['city'].isin(city_list)]
    df_sc = df_sc.merge(df_points[['point_id', 'city', TARGET_COL]].drop_duplicates(),
                        on=['point_id', 'city'], how='inner')
    df_sc = df_sc[df_sc[TARGET_COL] >= 0]
    print(f"  scene_card: {len(df_sc)} rows, {df_sc['point_id'].nunique()} buildings, {df_sc['city'].nunique()} cities")

    # parse date column (YYYYMMDD string or integer)
    df_sc['_plot_date'] = pd.to_datetime(df_sc['date'].astype(str), format='%Y%m%d')
    print(f"  CARD date range: {df_sc['_plot_date'].min().date()} to {df_sc['_plot_date'].max().date()}")

    # value column: s1__vv_mean per schema
    vv_col = 's1__vv_mean' if 's1__vv_mean' in df_sc.columns else None
    if vv_col is None:
        vv_candidates = [c for c in df_sc.columns if 'vv' in c.lower() and c.endswith('_mean')
                         and 'zscore' not in c and 'ratio' not in c and 'roll' not in c]
        vv_col = vv_candidates[0] if vv_candidates else None

    if vv_col:
        print(f"  Plotting: {vv_col}")
        plot_trajectories(df_sc, vv_col, '_plot_date', 'CARD VV')
    else:
        print("  SKIP: no VV mean column found")
        print(f"  Available columns: {[c for c in df_sc.columns if c.startswith('s1__')]}")
    del df_sc
except FileNotFoundError:
    print("  SKIP: no scene_card tier parquets found")

# COH trajectories
try:
    df_coh = load_v3_parquet('scene_coh')
    city_list = [c for c in CITIES_TO_PROCESS if c in df_coh['city'].unique()]
    df_coh = df_coh[df_coh['city'].isin(city_list)]
    df_coh = df_coh.merge(df_points[['point_id', 'city', TARGET_COL]].drop_duplicates(),
                          on=['point_id', 'city'], how='inner')
    df_coh = df_coh[df_coh[TARGET_COL] >= 0]
    print(f"  scene_coh: {len(df_coh)} rows, {df_coh['point_id'].nunique()} buildings, {df_coh['city'].nunique()} cities")

    # parse date1 column (YYYYMMDD string or integer) — COH uses date pairs
    df_coh['_plot_date'] = pd.to_datetime(df_coh['date1'].astype(str), format='%Y%m%d')
    print(f"  COH date range: {df_coh['_plot_date'].min().date()} to {df_coh['_plot_date'].max().date()}")

    # value column: s1__coh_vv_mean per schema
    coh_col = 's1__coh_vv_mean' if 's1__coh_vv_mean' in df_coh.columns else None
    if coh_col is None:
        coh_candidates = [c for c in df_coh.columns if 'coh' in c.lower() and c.endswith('_mean')
                          and 'zscore' not in c]
        coh_col = coh_candidates[0] if coh_candidates else None

    if coh_col:
        print(f"  Plotting: {coh_col}")
        plot_trajectories(df_coh, coh_col, '_plot_date', 'COH')
    else:
        print("  SKIP: no COH mean column found")
        print(f"  Available columns: {[c for c in df_coh.columns if c.startswith('s1__')]}")
    del df_coh
except FileNotFoundError:
    print("  SKIP: no scene_coh tier parquets found")

CELL D7: TEMPORAL TRAJECTORY VISUALIZATION
  Battle dates loaded for 21 cities
  scene_card: 834134 rows, 63243 buildings, 21 cities
  CARD date range: 2021-12-07 to 2026-02-23
  SKIP: no VV mean column found
  Available columns: ['s1__vv', 's1__vh']
  scene_coh: 468995 rows, 60769 buildings, 18 cities
  COH date range: 2021-12-07 to 2024-03-05
  SKIP: no COH mean column found
  Available columns: ['s1__coh_vv', 's1__coh_vh', 's1__coh_vv__zscore']


# CELL D8: TIER / PARQUET EXISTENCE DIAGNOSTIC# Why does NB09a show only 4 cities despite TIER_SELECTION=[0,1,2]?# Checks: file existence, row counts, damage label coverage per tier.

In [25]:
# @title CELL D8: V3 PARQUET INVENTORY
import gc

print("=" * 70)
print("CELL D8: V3 PARQUET INVENTORY")
print("=" * 70)

for pq_name, pq_info in sorted(MANIFEST['parquets'].items()):
    exists = any((V3_DIR / f"bda_{pq_name}_t{t}.parquet").exists() for t in _tiers)
    if not exists:
        print(f"  {pq_name:<35s} NOT ON DISK")
        continue
    fmt = pq_info.get('format', '?')
    n_feat = pq_info.get('n_features', 0)
    print(f"  {pq_name:<35s} [{pq_info['id']:>3s}] {fmt:>5s} {n_feat:>5d} feat")


CELL D8: V3 PARQUET INVENTORY
  block_accum_card                    [A27]  wide    15 feat
  block_accum_coh                     [A26]  wide     4 feat
  block_accum_ms                      [A28]  wide   168 feat
  block_stats                         [A15]  wide   106 feat
  card_drop                           [A19]  wide     7 feat
  coh_drop                            [A14]  wide     7 feat
  composite_prepost_bands             [ A9]  wide    79 feat
  composite_prepost_landuse           [A10]  wide     4 feat
  composite_vs_scenes_landuse         [A12]  long     3 feat
  fusion_card_cohdrop                 [ F3]  long     9 feat
  fusion_composite_blockstats         [ F8]  wide   185 feat
  fusion_composite_cohdrop            [ F7]  wide    86 feat
  fusion_indices_card                 [ F5]  long    11 feat
  fusion_indices_card_cohdrop         [ F6]  long    18 feat
  fusion_ms_card                      [ F1]  long    14 feat
  fusion_ms_card_cohdrop              [ F2]  long    21

# CELL D9: PARQUET CROSS-COMPARISON (product_prepost vs prepost_single)# Join both parquets on building_id for the same city.# Check if period-aggregated features correlate with simple single-scene features.# If not, period-aggregation is destroying signal.

In [26]:
# @title CELL D9: PARQUET COMPARISON (prepost_single_card vs block_stats)
# Goal: Compare simple pre/post CARD features (A13) vs full block stats (A15)
import gc

print("=" * 70)
print("CELL D9: PARQUET COMPARISON")
print("=" * 70)

# A13: prepost_single_card (12 features, all 21 cities, no NaN)
print("\n  --- A13: prepost_single_card ---")
df_a13, feat_a13, _ = get_analysis_df('prepost_single_card')
feat_a13 = exclude_leakage(feat_a13)
groups_a13 = df_a13['city'].values
y_a13 = df_a13[TARGET_COL].values
res_a13 = run_rf_cv(df_a13[feat_a13].values, y_a13, groups=groups_a13,
                    label=f"A13 prepost_single ({len(feat_a13)} feat, {df_a13['city'].nunique()} cities)")
del df_a13; gc.collect()

# A15: block_stats (900 features, 6 cities with data)
print("\n  --- A15: block_stats ---")
df_a15, feat_a15, _ = get_analysis_df('block_stats')
feat_a15_clean = exclude_leakage(feat_a15)
nan_pct = df_a15[feat_a15_clean].isna().mean()
feat_a15_clean = [c for c in feat_a15_clean if nan_pct[c] < 1.0]
groups_a15 = df_a15['city'].values
y_a15 = df_a15[TARGET_COL].values
res_a15 = run_rf_cv(df_a15[feat_a15_clean].values, y_a15, groups=groups_a15,
                    label=f"A15 block_stats ({len(feat_a15_clean)} feat, {df_a15['city'].nunique()} cities)")
del df_a15; gc.collect()

print(f"\n  Summary:")
print(f"    A13 (simple pre/post, 21 cities): AUC={res_a13['auc']:.3f}")
print(f"    A15 (block stats, partial cities): AUC={res_a15['auc']:.3f}")
print(f"    A13 is available for ALL cities. A15 is more powerful but limited to cities with COH SLC data.")
print("  Memory freed")


CELL D9: PARQUET COMPARISON

  --- A13: prepost_single_card ---
  Loaded prepost_single_card: 63243 rows, 6 features, 16.7 MB
    A13 prepost_single (6 feat, 21 cities)        AUC=0.514 F1=0.007 P=0.244 R=0.004 [GroupKFold]

  --- A15: block_stats ---
  Loaded block_stats: 63243 rows, 330 features, 96.6 MB
    A15 block_stats (324 feat, 21 cities)         AUC=0.533 F1=0.001 P=1.000 R=0.000 [GroupKFold]

  Summary:
    A13 (simple pre/post, 21 cities): AUC=0.514
    A15 (block stats, partial cities): AUC=0.533
    A13 is available for ALL cities. A15 is more powerful but limited to cities with COH SLC data.
  Memory freed


# CELL D10: FEATURE VALUE SANITY CHECK# Are product_prepost feature values in physically plausible ranges?# CARD backscatter should be dB (-30 to +5). COH should be 0-1.# If values are 0.0001-scale or 10000-scale, something is wrong.

In [27]:
# @title CELL D10: FEATURE VALUE SANITY CHECK (A9)
import gc

print("=" * 70)
print("CELL D10: FEATURE VALUE SANITY CHECK")
print("  Parquet: composite_prepost_bands (A9)")
print("=" * 70)

df_a9, feat_a9, _ = get_analysis_df('composite_prepost_bands')

# expected physical ranges
EXPECTED_RANGES = {
    's1__': {'min': -35.0, 'max': 10.0, 'desc': 'backscatter dB'},
    's1__coh': {'min': 0.0, 'max': 1.0, 'desc': 'coherence'},
    's2__composite__b': {'min': 0.0, 'max': 1.0, 'desc': 'reflectance'},
    's2__composite__nd': {'min': -1.0, 'max': 1.0, 'desc': 'spectral index'},
    'landuse': {'min': 0.0, 'max': 10.0, 'desc': 'class code'},
}

print(f"\n  Checking {len(feat_a9)} features against physical ranges:")
anomalies = 0
for c in feat_a9:
    vals = df_a9[c].dropna().values
    if len(vals) < 20:
        continue
    vmin, vmax = float(vals.min()), float(vals.max())
    for prefix, expected in EXPECTED_RANGES.items():
        if c.startswith(prefix):
            if vmin < expected['min'] - 10 or vmax > expected['max'] + 10:
                print(f"    ANOMALY: {c:55s} [{vmin:.3f}, {vmax:.3f}] expected {expected['desc']}")
                anomalies += 1
            break

if anomalies == 0:
    print(f"    All features within expected physical ranges")

# constant features
dead = [c for c in feat_a9 if df_a9[c].dropna().std() < 1e-6 and df_a9[c].notna().sum() > 20]
if dead:
    print(f"\n  {len(dead)} constant features (std < 1e-6): {dead[:5]}")
else:
    print(f"\n  No constant features (good)")

del df_a9; gc.collect()
print("  Memory freed")


CELL D10: FEATURE VALUE SANITY CHECK
  Parquet: composite_prepost_bands (A9)
  Loaded composite_prepost_bands: 62043 rows, 79 features, 32.5 MB

  Checking 79 features against physical ranges:
    All features within expected physical ranges

  No constant features (good)
  Memory freed


In [28]:
# @title CELL D11: SAVE NB07 RESULTS
results_df = pd.DataFrame(NB07_RESULTS)
if len(results_df) > 0:
    save_result(results_df, 'nb07_experiment_results', '')
    print(f"  {len(results_df)} experiments logged")
    print(results_df[['experiment', 'parquet', 'cv_method', 'auc', 'f1', 'n_features', 'n_cities']].to_string(index=False))
else:
    print("  No experiments logged")

# OOF predictions inventory (for NB12 overlap analysis)
print(f"\n  OOF predictions inventory  ({OOF_DIR})")
oof_files = sorted(OOF_DIR.glob('oof_*.parquet'))
if oof_files:
    inv_rows = []
    for p in oof_files:
        head = pd.read_parquet(p, columns=['model_id', 'variant_id', 'cm_class'])
        cm = head['cm_class'].value_counts()
        inv_rows.append({
            'file':       p.name,
            'model_id':   head['model_id'].iloc[0],
            'variant_id': head['variant_id'].iloc[0],
            'n_rows':     len(head),
            'TP':         int(cm.get('TP', 0)),
            'FN':         int(cm.get('FN', 0)),
            'FP':         int(cm.get('FP', 0)),
            'TN':         int(cm.get('TN', 0)),
        })
    inv_df = pd.DataFrame(inv_rows)
    save_result(inv_df, 'nb07_oof_inventory', '')
    print(inv_df.to_string(index=False))
    print(f"\n  Total OOF parquets: {len(oof_files)}")
else:
    print("  No OOF parquets saved")


    saved -> nb07_experiment_results.csv (39 rows)
  39 experiments logged
              experiment                     parquet       cv_method      auc       f1  n_features  n_cities
             D1_pre_only         prepost_single_card      GroupKFold 0.505354 0.029838           2        21
            D1_post_only         prepost_single_card      GroupKFold 0.505781 0.026860           2        21
           D1_delta_only         prepost_single_card      GroupKFold 0.512269 0.028086           2        21
             D1_pre+post         prepost_single_card      GroupKFold 0.503369 0.004792           4        21
       D1_pre+post+delta         prepost_single_card      GroupKFold 0.514450 0.007882           6        21
    D1_pre_baseline_mean                 block_stats      GroupKFold 0.498549 0.026972           2        21
      D1_post_block_mean                 block_stats      GroupKFold 0.498549 0.026972           2        21
     D1_pre_baseline_all                 block_stats 